# AQUA — Model Interpretation and Error Analysis

## 1. Introduction

The previous notebook developed and evaluated Logistic Regression and Random Forest models for predicting the water-quality status at the next recorded observation for an aquaculture pond.

Although the models demonstrated predictive capability, their evaluation metrics alone do not explain how they behave under different aquaculture conditions.

A model may correctly classify many observations while consistently missing particular types of water-quality problems. Similarly, frequent false alerts may reduce its practical usefulness.

This notebook investigates the behaviour, prediction errors and interpretation of the existing models.

## 2. Research Objective

The primary objective is to understand where, why and under what conditions the predictive models produce correct and incorrect water-quality predictions.

The analysis will focus on:

1. Understanding false-positive and false-negative predictions.
2. Identifying ponds with repeated prediction errors.
3. Examining model performance across morning and later-day measurements.
4. Investigating the relationship between prediction errors and irregular observation intervals.
5. Understanding the contribution of pH, dissolved oxygen and temperature to model predictions.
6. Identifying patterns that may explain differences in model behaviour between Eluru and Nellore.

## 3. Scope of Analysis

The analysis will use the existing fitted Logistic Regression and Random Forest models and their corresponding regional test datasets.

The three initial predictors remain:

- pH
- Dissolved oxygen (DO)
- Temperature

The regional models and their previously selected operating thresholds will be retained for error analysis.

The existing training and test partitions will remain unchanged.

No new model will be selected or tuned using the final test observations.

## 4. Research Questions

This notebook will investigate the following questions:

- Which actual out-of-range observations are missed by the models?
- What characteristics distinguish false-positive observations from correctly identified cases?
- Do prediction errors differ between morning and later-day measurements?
- Are longer intervals between consecutive observations associated with different prediction outcomes?
- Which predictors contribute most to the models' decisions?
- Are model errors concentrated in particular ponds or environmental conditions?
- Do the patterns differ between Eluru and Nellore?

## 5. Expected Outcome

The expected outcome is a scientifically interpretable assessment of model behaviour, identifying strengths, weaknesses and conditions under which predictions may become unreliable.

The findings will inform subsequent robustness assessment, probability calibration and final model development.

## 6. Methodological Precautions

- Existing test observations will be used for retrospective error analysis, not repeated model tuning.
- Repeated observations from the same pond will be treated as clustered data.
- Associations will not automatically be interpreted as causal relationships.
- Morning and later-day dissolved oxygen measurements will be interpreted in their measurement context.
- Eluru and Nellore findings will be reported separately where their data coverage or behaviour differs.
- Model interpretation will distinguish predictive importance from biological or causal importance.

In [1]:
# CELL 1: PROJECT SETUP AND DATA AVAILABILITY CHECK

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --------------------------------------------------
# 1. Locate the AQUA project directory
# --------------------------------------------------

possible_roots = [
    Path.cwd(),
    Path.cwd().parent
]

project_root = next(
    (
        path.resolve()
        for path in possible_roots
        if (path / "data" / "raw" / "water_quality.csv").is_file()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate AQUA project directory. "
        "Expected data/raw/water_quality.csv."
    )

# --------------------------------------------------
# 2. Define project directories
# --------------------------------------------------

data_directory = project_root / "data" / "raw"
tables_directory = project_root / "outputs" / "tables"

# --------------------------------------------------
# 3. Load the original dataset
# --------------------------------------------------

raw_data_path = data_directory / "water_quality.csv"

aqua_df = pd.read_csv(raw_data_path)

# Verify essential columns.
required_columns = [
    "pond_id",
    "region",
    "Date of data collection",
    "Time of data collection",
    "pH",
    "DO (mg/L)",
    "Temp (in °C)",
    "Is WQ in range?"
]

missing_columns = [
    column for column in required_columns
    if column not in aqua_df.columns
]

if missing_columns:
    raise KeyError(
        f"Required columns missing from dataset: {missing_columns}"
    )

# --------------------------------------------------
# 4. Check previously saved modelling outputs
# --------------------------------------------------

expected_artifacts = {
    "Regional model comparison":
        tables_directory / "regional_model_comparison.csv",

    "Regional bootstrap comparison":
        tables_directory / "regional_bootstrap_comparison.csv",

    "Final modelling consistency audit":
        tables_directory / "final_modelling_consistency_audit.csv"
}

inventory = []

inventory.append({
    "Artifact": "Original water-quality dataset",
    "Status": "AVAILABLE",
    "Details": f"{len(aqua_df):,} rows × {len(aqua_df.columns)} columns"
})

for artifact_name, artifact_path in expected_artifacts.items():

    exists = artifact_path.is_file()

    inventory.append({
        "Artifact": artifact_name,
        "Status": "AVAILABLE" if exists else "NOT FOUND",
        "Details": str(artifact_path.relative_to(project_root))
    })

# --------------------------------------------------
# 5. Display one consolidated availability table
# --------------------------------------------------

artifact_inventory = pd.DataFrame(inventory)

display(artifact_inventory)

,Artifact,Status,Details
0,Original water-quality dataset,AVAILABLE,"12,431 rows × 53 columns"
1,Regional model comparison,AVAILABLE,outputs\tables\regional_model_comparison.csv
2,Regional bootstrap comparison,AVAILABLE,outputs\tables\regional_bootstrap_comparison.csv
3,Final modelling consistency audit,AVAILABLE,outputs\tables\final_modelling_consistency_aud...


In [2]:
# CELL 2: LOAD AND VERIFY NOTEBOOK 03 MODELLING ARTIFACTS

import json
import joblib

# --------------------------------------------------
# 1. Locate exported artifacts
# --------------------------------------------------

artifact_directory = project_root / "outputs" / "model_artifacts"

prediction_file = (
    artifact_directory
    / "regional_test_predictions_for_error_analysis.csv"
)

model_file = (
    artifact_directory
    / "regional_three_feature_models.joblib"
)

manifest_file = artifact_directory / "model_manifest.json"

# --------------------------------------------------
# 2. Load prediction records, models and manifest
# --------------------------------------------------

error_analysis_data = pd.read_csv(prediction_file)

model_bundle_04 = joblib.load(model_file)

with open(manifest_file, "r", encoding="utf-8") as file:
    model_manifest = json.load(file)

# --------------------------------------------------
# 3. Verify prediction record structure
# --------------------------------------------------

required_columns = [
    "Region",
    "pond_id",
    "Actual status",
    "Actual class",
    "pH",
    "DO (mg/L)",
    "Temp (in °C)",
    "LR probability",
    "LR threshold",
    "LR predicted class",
    "RF probability",
    "RF threshold",
    "RF predicted class",
    "LR correct",
    "RF correct"
]

missing_columns = [
    column for column in required_columns
    if column not in error_analysis_data.columns
]

if missing_columns:
    raise ValueError(
        f"Missing prediction columns: {missing_columns}"
    )

# --------------------------------------------------
# 4. Validate regional record counts
# --------------------------------------------------

expected_counts = {
    "Eluru": 2232,
    "Nellore": 573
}

actual_counts = (
    error_analysis_data["Region"]
    .value_counts()
    .to_dict()
)

count_check = actual_counts == expected_counts

# --------------------------------------------------
# 5. Verify prediction classes against probabilities
# --------------------------------------------------

lr_recalculated = (
    error_analysis_data["LR probability"]
    >= error_analysis_data["LR threshold"]
).astype(int)

rf_recalculated = (
    error_analysis_data["RF probability"]
    >= error_analysis_data["RF threshold"]
).astype(int)

lr_prediction_check = np.array_equal(
    lr_recalculated.to_numpy(),
    error_analysis_data["LR predicted class"].to_numpy()
)

rf_prediction_check = np.array_equal(
    rf_recalculated.to_numpy(),
    error_analysis_data["RF predicted class"].to_numpy()
)

# Verify correctness indicators.
lr_correct_check = np.array_equal(
    (
        error_analysis_data["Actual class"]
        == error_analysis_data["LR predicted class"]
    ).to_numpy(),
    error_analysis_data["LR correct"].to_numpy()
)

rf_correct_check = np.array_equal(
    (
        error_analysis_data["Actual class"]
        == error_analysis_data["RF predicted class"]
    ).to_numpy(),
    error_analysis_data["RF correct"].to_numpy()
)

# --------------------------------------------------
# 6. Verify fitted model availability and predictors
# --------------------------------------------------

expected_models = {
    "Eluru": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Nellore": [
        "Logistic Regression",
        "Random Forest"
    ]
}

model_checks = []

for region, model_names in expected_models.items():

    for model_name in model_names:

        model = model_bundle_04[region][model_name]

        if model_name == "Logistic Regression":
            feature_count = (
                model.named_steps["model"].n_features_in_
            )
        else:
            feature_count = model.n_features_in_

        model_checks.append(
            feature_count == len(model_manifest["predictors"])
        )

all_models_available = all(model_checks)

# --------------------------------------------------
# 7. Produce one consolidated verification table
# --------------------------------------------------

verification_results = [
    {
        "Verification": "Prediction records loaded",
        "Result": "PASS",
        "Details": f"{len(error_analysis_data):,} records"
    },
    {
        "Verification": "Regional observation counts",
        "Result": "PASS" if count_check else "REVIEW",
        "Details": str(actual_counts)
    },
    {
        "Verification": "Required prediction columns",
        "Result": "PASS",
        "Details": f"{len(required_columns)} verified columns"
    },
    {
        "Verification": "Logistic Regression predictions",
        "Result": "PASS" if lr_prediction_check else "REVIEW",
        "Details": "Recalculated predictions match exported predictions"
    },
    {
        "Verification": "Random Forest predictions",
        "Result": "PASS" if rf_prediction_check else "REVIEW",
        "Details": "Recalculated predictions match exported predictions"
    },
    {
        "Verification": "Logistic Regression correctness flags",
        "Result": "PASS" if lr_correct_check else "REVIEW",
        "Details": "Correctness indicators verified"
    },
    {
        "Verification": "Random Forest correctness flags",
        "Result": "PASS" if rf_correct_check else "REVIEW",
        "Details": "Correctness indicators verified"
    },
    {
        "Verification": "Four fitted models loaded",
        "Result": "PASS" if all_models_available else "REVIEW",
        "Details": "Three-feature models verified for both regions"
    },
    {
        "Verification": "Model manifest loaded",
        "Result": "PASS",
        "Details": str(model_manifest["predictors"])
    }
]

artifact_verification = pd.DataFrame(verification_results)

display(artifact_verification)

,Verification,Result,Details
0,Prediction records loaded,PASS,"2,805 records"
1,Regional observation counts,PASS,"{'Eluru': 2232, 'Nellore': 573}"
2,Required prediction columns,PASS,15 verified columns
3,Logistic Regression predictions,PASS,Recalculated predictions match exported predic...
4,Random Forest predictions,PASS,Recalculated predictions match exported predic...
5,Logistic Regression correctness flags,PASS,Correctness indicators verified
6,Random Forest correctness flags,PASS,Correctness indicators verified
7,Four fitted models loaded,PASS,Three-feature models verified for both regions
8,Model manifest loaded,PASS,"['pH', 'DO (mg/L)', 'Temp (in °C)']"


In [3]:
# CELL 3: REGIONAL PREDICTION ERROR PROFILE

error_categories = [
    "True Positive",
    "False Negative",
    "False Positive",
    "True Negative"
]

error_profile_rows = []

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

# --------------------------------------------------
# 1. Examine each region and model
# --------------------------------------------------

for region in ["Eluru", "Nellore"]:

    region_data = error_analysis_data[
        error_analysis_data["Region"] == region
    ].copy()

    for model_name, prediction_column in model_columns.items():

        actual = region_data["Actual class"].astype(int)
        predicted = region_data[prediction_column].astype(int)

        outcome_masks = {
            "True Positive": (actual == 1) & (predicted == 1),
            "False Negative": (actual == 1) & (predicted == 0),
            "False Positive": (actual == 0) & (predicted == 1),
            "True Negative": (actual == 0) & (predicted == 0)
        }

        for category in error_categories:

            selected = outcome_masks[category]
            selected_data = region_data.loc[selected]

            count = int(selected.sum())

            error_profile_rows.append({
                "Region": region,
                "Model": model_name,
                "Prediction outcome": category,
                "Observations": count,
                "Percentage of test observations": (
                    count / len(region_data) * 100
                ),
                "Unique ponds": selected_data["pond_id"].nunique()
            })

# --------------------------------------------------
# 2. Create one consolidated results table
# --------------------------------------------------

regional_error_profile = pd.DataFrame(error_profile_rows)

regional_error_profile[
    "Percentage of test observations"
] = regional_error_profile[
    "Percentage of test observations"
].round(2)

display(regional_error_profile)

,Region,Model,Prediction outcome,Observations,Percentage of test observations,Unique ponds
0,Eluru,Logistic Regression,True Positive,191,8.56,57
1,Eluru,Logistic Regression,False Negative,121,5.42,57
2,Eluru,Logistic Regression,False Positive,863,38.66,116
3,Eluru,Logistic Regression,True Negative,1057,47.36,113
4,Eluru,Random Forest,True Positive,163,7.30,54
5,Eluru,Random Forest,False Negative,149,6.68,67
6,Eluru,Random Forest,False Positive,351,15.73,103
7,Eluru,Random Forest,True Negative,1569,70.30,118
8,Nellore,Logistic Regression,True Positive,17,2.97,11
9,Nellore,Logistic Regression,False Negative,27,4.71,17


In [4]:
# CELL 4: POND-LEVEL FALSE POSITIVE AND FALSE NEGATIVE ANALYSIS

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

pond_error_records = []

for region in ["Eluru", "Nellore"]:

    region_data = error_analysis_data[
        error_analysis_data["Region"] == region
    ].copy()

    for model_name, prediction_column in model_columns.items():

        # Calculate error counts separately for each pond.
        pond_summary = region_data.groupby("pond_id").apply(
            lambda group: pd.Series({

                "Observations": len(group),

                "Actual out-of-range": (
                    group["Actual class"] == 1
                ).sum(),

                "Actual in-range": (
                    group["Actual class"] == 0
                ).sum(),

                "False negatives": (
                    (group["Actual class"] == 1)
                    & (group[prediction_column] == 0)
                ).sum(),

                "False positives": (
                    (group["Actual class"] == 0)
                    & (group[prediction_column] == 1)
                ).sum()

            }),
            include_groups=False
        ).reset_index()

        # Calculate error rates within each pond.
        pond_summary["False negative rate"] = (
            pond_summary["False negatives"]
            / pond_summary["Actual out-of-range"]
        )

        pond_summary["False positive rate"] = (
            pond_summary["False positives"]
            / pond_summary["Actual in-range"]
        )

        # Summarise the distribution of pond-level errors.
        positive_ponds = pond_summary[
            pond_summary["Actual out-of-range"] > 0
        ]

        negative_ponds = pond_summary[
            pond_summary["Actual in-range"] > 0
        ]

        pond_error_records.append({

            "Region": region,
            "Model": model_name,

            "Test ponds": len(pond_summary),

            "Ponds with out-of-range observations": len(
                positive_ponds
            ),

            "Ponds with missed out-of-range observations": (
                positive_ponds["False negatives"] > 0
            ).sum(),

            "Median pond-level false negative rate (%)": (
                positive_ponds["False negative rate"].median() * 100
            ),

            "Ponds with false alerts": (
                negative_ponds["False positives"] > 0
            ).sum(),

            "Median pond-level false positive rate (%)": (
                negative_ponds["False positive rate"].median() * 100
            )

        })

# Create one consolidated output table.
pond_error_concentration = pd.DataFrame(
    pond_error_records
)

percentage_columns = [
    "Median pond-level false negative rate (%)",
    "Median pond-level false positive rate (%)"
]

pond_error_concentration[percentage_columns] = (
    pond_error_concentration[percentage_columns].round(2)
)

display(pond_error_concentration)

,Region,Model,Test ponds,Ponds with out-of-range observations,Ponds with missed out-of-range observations,Median pond-level false negative rate (%),Ponds with false alerts,Median pond-level false positive rate (%)
0,Eluru,Logistic Regression,118,70,57,50.0,116,46.61
1,Eluru,Random Forest,118,70,67,50.0,103,18.35
2,Nellore,Logistic Regression,49,19,17,50.0,8,0.00
3,Nellore,Random Forest,49,19,16,50.0,18,0.00


In [5]:
# CELL 5: DISTRIBUTION OF POND-LEVEL ERROR RATES

pond_distribution_rows = []

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

for region in ["Eluru", "Nellore"]:

    region_data = error_analysis_data[
        error_analysis_data["Region"] == region
    ].copy()

    for model_name, prediction_column in model_columns.items():

        # Aggregate prediction outcomes within each pond.
        pond_summary = region_data.groupby("pond_id").apply(
            lambda group: pd.Series({

                "Actual out-of-range": (
                    group["Actual class"] == 1
                ).sum(),

                "Actual in-range": (
                    group["Actual class"] == 0
                ).sum(),

                "False negatives": (
                    (group["Actual class"] == 1)
                    & (group[prediction_column] == 0)
                ).sum(),

                "False positives": (
                    (group["Actual class"] == 0)
                    & (group[prediction_column] == 1)
                ).sum()

            }),
            include_groups=False
        )

        # Calculate error rates using the appropriate denominators.
        pond_summary["False negative rate"] = (
            pond_summary["False negatives"]
            / pond_summary["Actual out-of-range"]
        )

        pond_summary["False positive rate"] = (
            pond_summary["False positives"]
            / pond_summary["Actual in-range"]
        )

        fn_rates = pond_summary.loc[
            pond_summary["Actual out-of-range"] > 0,
            "False negative rate"
        ]

        fp_rates = pond_summary.loc[
            pond_summary["Actual in-range"] > 0,
            "False positive rate"
        ]

        # Summarise both types of error.
        for error_type, rates in [
            ("False negative rate", fn_rates),
            ("False positive rate", fp_rates)
        ]:

            pond_distribution_rows.append({

                "Region": region,
                "Model": model_name,
                "Error type": error_type,

                "Eligible ponds": len(rates),

                "Ponds with errors": int((rates > 0).sum()),

                "Mean error rate (%)": rates.mean() * 100,

                "25th percentile (%)": rates.quantile(0.25) * 100,

                "Median error rate (%)": rates.median() * 100,

                "75th percentile (%)": rates.quantile(0.75) * 100,

                "Maximum error rate (%)": rates.max() * 100

            })

pond_error_distribution = pd.DataFrame(
    pond_distribution_rows
)

percentage_columns = [
    "Mean error rate (%)",
    "25th percentile (%)",
    "Median error rate (%)",
    "75th percentile (%)",
    "Maximum error rate (%)"
]

pond_error_distribution[percentage_columns] = (
    pond_error_distribution[percentage_columns].round(2)
)

display(pond_error_distribution)

,Region,Model,Error type,Eligible ponds,Ponds with errors,Mean error rate (%),25th percentile (%),Median error rate (%),75th percentile (%),Maximum error rate (%)
0,Eluru,Logistic Regression,False negative rate,70,57,45.95,21.25,50.00,70.24,100.00
1,Eluru,Logistic Regression,False positive rate,118,116,46.78,36.44,46.61,57.14,100.00
2,Eluru,Random Forest,False negative rate,70,67,57.57,34.38,50.00,75.00,100.00
3,Eluru,Random Forest,False positive rate,118,103,17.92,8.33,18.35,25.00,66.67
4,Nellore,Logistic Regression,False negative rate,19,17,65.39,50.00,50.00,100.00,100.00
5,Nellore,Logistic Regression,False positive rate,49,8,1.65,0.00,0.00,0.00,28.57
6,Nellore,Random Forest,False negative rate,19,16,51.40,50.00,50.00,55.00,100.00
7,Nellore,Random Forest,False positive rate,49,18,4.13,0.00,0.00,6.67,33.33


In [7]:
# CELL 7: CORRECTED VERSION OF CELL 6 — POND-LEVEL MISSED PREDICTION ANALYSIS
# CORRECTED CELL: IDENTIFY PONDS WITH THE MOST MISSED PREDICTIONS

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

pond_miss_records = []

for region in ["Eluru", "Nellore"]:

    region_data = error_analysis_data[
        error_analysis_data["Region"] == region
    ].copy()

    for model_name, prediction_column in model_columns.items():

        region_data["Missed problem"] = (
            (region_data["Actual class"] == 1)
            & (region_data[prediction_column] == 0)
        )

        # Aggregate observations and missed predictions per pond.
        pond_summary = (
            region_data.groupby("pond_id")
            .agg(
                Total_observations=("Actual class", "size"),
                Actual_out_of_range=("Actual class", "sum"),
                Missed_observations=("Missed problem", "sum")
            )
            .reset_index()
        )

        # Keep ponds with at least one actual out-of-range observation.
        pond_summary = pond_summary[
            pond_summary["Actual_out_of_range"] > 0
        ].copy()

        pond_summary["Miss_rate_percent"] = (
            pond_summary["Missed_observations"]
            / pond_summary["Actual_out_of_range"]
            * 100
        )

        pond_summary["Region"] = region
        pond_summary["Model"] = model_name

        pond_miss_records.append(pond_summary)

# Combine all results.
pond_miss_analysis = pd.concat(
    pond_miss_records,
    ignore_index=True
)

# Select the five ponds with the most missed predictions
# within each region and model.
top_missed_ponds = (
    pond_miss_analysis
    .sort_values(
        [
            "Region",
            "Model",
            "Missed_observations",
            "Miss_rate_percent"
        ],
        ascending=[True, True, False, False]
    )
    .groupby(["Region", "Model"], group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

# Format output column names.
top_missed_ponds = top_missed_ponds.rename(columns={
    "Total_observations": "Total observations",
    "Actual_out_of_range": "Actual out-of-range",
    "Missed_observations": "Missed observations",
    "Miss_rate_percent": "Miss rate (%)"
})

top_missed_ponds["Miss rate (%)"] = (
    top_missed_ponds["Miss rate (%)"].round(2)
)

display(
    top_missed_ponds[
        [
            "Region",
            "Model",
            "pond_id",
            "Total observations",
            "Actual out-of-range",
            "Missed observations",
            "Miss rate (%)"
        ]
    ]
)

,Region,Model,pond_id,Total observations,Actual out-of-range,Missed observations,Miss rate (%)
0,Eluru,Logistic Regression,ara2_31afc592,33,11,8,72.73
1,Eluru,Logistic Regression,ara2_46286ebb,34,13,8,61.54
2,Eluru,Logistic Regression,ara2_b0a5ed5f,24,6,6,100.00
3,Eluru,Logistic Regression,ara2_f5124b92,25,5,5,100.00
4,Eluru,Logistic Regression,ara2_e6fe1327,39,7,5,71.43
5,Eluru,Random Forest,ara2_00793177,38,16,8,50.00
6,Eluru,Random Forest,ara2_31afc592,33,11,7,63.64
7,Eluru,Random Forest,ara2_46286ebb,34,13,7,53.85
8,Eluru,Random Forest,ara2_f5dabd1c,33,6,6,100.00
9,Eluru,Random Forest,ara2_d8f6a2aa,38,9,5,55.56


In [8]:
# CELL 8: WATER-QUALITY PROFILE OF DETECTED AND MISSED PROBLEMS

predictors = [
    "pH",
    "DO (mg/L)",
    "Temp (in °C)"
]

model_configurations = {
    "Logistic Regression": {
        "prediction_column": "LR predicted class",
        "probability_column": "LR probability"
    },
    "Random Forest": {
        "prediction_column": "RF predicted class",
        "probability_column": "RF probability"
    }
}

profile_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = error_analysis_data[
        error_analysis_data["Region"] == region
    ].copy()

    # Consider only observations that were actually out of range.
    positive_cases = region_data[
        region_data["Actual class"] == 1
    ].copy()

    for model_name, config in model_configurations.items():

        prediction_column = config["prediction_column"]
        probability_column = config["probability_column"]

        # Separate correctly detected and missed cases.
        positive_cases["Detection outcome"] = np.where(
            positive_cases[prediction_column] == 1,
            "Detected (TP)",
            "Missed (FN)"
        )

        for outcome, group in positive_cases.groupby(
            "Detection outcome"
        ):

            for feature in predictors:

                values = pd.to_numeric(
                    group[feature],
                    errors="coerce"
                ).dropna()

                profile_rows.append({

                    "Region": region,
                    "Model": model_name,
                    "Detection outcome": outcome,
                    "Parameter": feature,

                    "Observations": len(values),

                    "Unique ponds": group.loc[
                        values.index, "pond_id"
                    ].nunique(),

                    "Mean": values.mean(),
                    "Median": values.median(),
                    "25th percentile": values.quantile(0.25),
                    "75th percentile": values.quantile(0.75)

                })

# Create one consolidated descriptive table.
missed_condition_profile = pd.DataFrame(profile_rows)

numeric_columns = [
    "Mean",
    "Median",
    "25th percentile",
    "75th percentile"
]

missed_condition_profile[numeric_columns] = (
    missed_condition_profile[numeric_columns].round(3)
)

display(missed_condition_profile)

,Region,Model,Detection outcome,Parameter,Observations,Unique ponds,Mean,Median,25th percentile,75th percentile
0,Eluru,Logistic Regression,Detected (TP),pH,191,57,8.487,8.45,8.290,8.650
1,Eluru,Logistic Regression,Detected (TP),DO (mg/L),191,57,5.077,3.30,2.400,8.300
2,Eluru,Logistic Regression,Detected (TP),Temp (in °C),191,57,29.404,29.60,28.450,30.750
3,Eluru,Logistic Regression,Missed (FN),pH,121,57,8.350,8.39,8.290,8.450
4,Eluru,Logistic Regression,Missed (FN),DO (mg/L),121,57,10.343,11.10,9.700,11.600
5,Eluru,Logistic Regression,Missed (FN),Temp (in °C),121,57,28.990,28.60,27.200,31.500
6,Eluru,Random Forest,Detected (TP),pH,163,54,8.521,8.49,8.380,8.680
7,Eluru,Random Forest,Detected (TP),DO (mg/L),163,54,6.044,3.40,2.150,10.900
8,Eluru,Random Forest,Detected (TP),Temp (in °C),163,54,29.282,29.60,27.900,30.950
9,Eluru,Random Forest,Missed (FN),pH,149,67,8.338,8.36,8.250,8.430


In [9]:
# CELL 9: MEASUREMENT-TIME ANALYSIS OF DETECTED AND MISSED CASES

# Verify that the earlier observation timestamp was exported.
if "Observation timestamp" not in error_analysis_data.columns:
    raise KeyError(
        "Observation timestamp is unavailable in the exported predictions."
    )

time_analysis_data = error_analysis_data.copy()

# Parse the earlier observation timestamp.
time_analysis_data["Observation timestamp"] = pd.to_datetime(
    time_analysis_data["Observation timestamp"],
    errors="coerce"
)

# Extract the hour of measurement.
time_analysis_data["Measurement hour"] = (
    time_analysis_data["Observation timestamp"].dt.hour
)

# Retain only actual out-of-range observations.
time_analysis_data = time_analysis_data[
    time_analysis_data["Actual class"] == 1
].copy()

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

hourly_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = time_analysis_data[
        time_analysis_data["Region"] == region
    ]

    for model_name, prediction_column in model_columns.items():

        region_data = region_data.copy()

        region_data["Detection outcome"] = np.where(
            region_data[prediction_column] == 1,
            "Detected (TP)",
            "Missed (FN)"
        )

        hourly_summary = (
            region_data.dropna(subset=["Measurement hour"])
            .groupby(
                ["Detection outcome", "Measurement hour"]
            )
            .agg(
                Observations=("pond_id", "size"),
                Unique_ponds=("pond_id", "nunique"),
                Median_DO=("DO (mg/L)", "median")
            )
            .reset_index()
        )

        hourly_summary["Region"] = region
        hourly_summary["Model"] = model_name

        hourly_rows.append(hourly_summary)

hourly_detection_analysis = pd.concat(
    hourly_rows,
    ignore_index=True
)

hourly_detection_analysis = hourly_detection_analysis[
    [
        "Region",
        "Model",
        "Detection outcome",
        "Measurement hour",
        "Observations",
        "Unique_ponds",
        "Median_DO"
    ]
].rename(columns={
    "Unique_ponds": "Unique ponds",
    "Median_DO": "Median DO (mg/L)"
})

hourly_detection_analysis["Median DO (mg/L)"] = (
    hourly_detection_analysis["Median DO (mg/L)"].round(2)
)

display(
    hourly_detection_analysis.sort_values(
        ["Region", "Model", "Detection outcome", "Measurement hour"]
    ).reset_index(drop=True)
)

,Region,Model,Detection outcome,Measurement hour,Observations,Unique ponds,Median DO (mg/L)
0,Eluru,Logistic Regression,Detected (TP),6,15,12,3.00
1,Eluru,Logistic Regression,Detected (TP),7,114,52,3.00
2,Eluru,Logistic Regression,Detected (TP),8,12,9,3.10
3,Eluru,Logistic Regression,Detected (TP),15,8,5,11.25
4,Eluru,Logistic Regression,Detected (TP),16,42,21,11.35
5,Eluru,Logistic Regression,Missed (FN),6,2,2,3.55
6,Eluru,Logistic Regression,Missed (FN),7,12,7,2.65
7,Eluru,Logistic Regression,Missed (FN),8,2,2,3.10
8,Eluru,Logistic Regression,Missed (FN),15,39,27,10.90
9,Eluru,Logistic Regression,Missed (FN),16,66,41,11.30


In [10]:
# CELL 10: DETECTION PERFORMANCE BY EARLIER MEASUREMENT PERIOD

# Use actual out-of-range observations only.
positive_cases = time_analysis_data.copy()

# Group recorded measurement hours.
positive_cases["Measurement period"] = np.select(
    [
        positive_cases["Measurement hour"].between(6, 8),
        positive_cases["Measurement hour"].between(15, 17)
    ],
    [
        "Morning (06:00–08:59)",
        "Later day (15:00–17:59)"
    ],
    default="Other hours"
)

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

period_results = []

for region in ["Eluru", "Nellore"]:

    region_data = positive_cases[
        positive_cases["Region"] == region
    ]

    for model_name, prediction_column in model_columns.items():

        for period in [
            "Morning (06:00–08:59)",
            "Later day (15:00–17:59)",
            "Other hours"
        ]:

            period_data = region_data[
                region_data["Measurement period"] == period
            ]

            if len(period_data) == 0:
                continue

            detected = (
                period_data[prediction_column] == 1
            ).sum()

            missed = (
                period_data[prediction_column] == 0
            ).sum()

            total = len(period_data)

            period_results.append({

                "Region": region,
                "Model": model_name,
                "Measurement period": period,

                "Actual out-of-range cases": total,
                "Detected": int(detected),
                "Missed": int(missed),

                "Recall (%)": round(
                    detected / total * 100, 2
                ),

                "Miss rate (%)": round(
                    missed / total * 100, 2
                ),

                "Unique ponds": period_data["pond_id"].nunique()

            })

period_detection_summary = pd.DataFrame(period_results)

display(period_detection_summary)

,Region,Model,Measurement period,Actual out-of-range cases,Detected,Missed,Recall (%),Miss rate (%),Unique ponds
0,Eluru,Logistic Regression,Morning (06:00–08:59),157,141,16,89.81,10.19,59
1,Eluru,Logistic Regression,Later day (15:00–17:59),155,50,105,32.26,67.74,62
2,Eluru,Random Forest,Morning (06:00–08:59),157,102,55,64.97,35.03,59
3,Eluru,Random Forest,Later day (15:00–17:59),155,61,94,39.35,60.65,62
4,Nellore,Logistic Regression,Morning (06:00–08:59),22,14,8,63.64,36.36,17
5,Nellore,Logistic Regression,Later day (15:00–17:59),22,3,19,13.64,86.36,16
6,Nellore,Random Forest,Morning (06:00–08:59),22,21,1,95.45,4.55,17
7,Nellore,Random Forest,Later day (15:00–17:59),22,0,22,0.00,100.00,16


In [11]:
# CELL 11: WATER-QUALITY CONDITIONS WITHIN MEASUREMENT PERIODS

period_data = error_analysis_data.copy()

# Classify measurement periods using the earlier observation timestamp.
period_data["Observation timestamp"] = pd.to_datetime(
    period_data["Observation timestamp"],
    errors="coerce"
)

period_data["Measurement hour"] = (
    period_data["Observation timestamp"].dt.hour
)

period_data["Measurement period"] = np.select(
    [
        period_data["Measurement hour"].between(6, 8),
        period_data["Measurement hour"].between(15, 17)
    ],
    [
        "Morning",
        "Later day"
    ],
    default="Other hours"
)

model_configurations = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

condition_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = period_data[
        (period_data["Region"] == region)
        & (period_data["Actual class"] == 1)
    ].copy()

    for model_name, prediction_column in model_configurations.items():

        region_data["Detection outcome"] = np.where(
            region_data[prediction_column] == 1,
            "Detected (TP)",
            "Missed (FN)"
        )

        summary = (
            region_data
            .groupby(
                ["Measurement period", "Detection outcome"],
                observed=True
            )
            .agg(
                Observations=("pond_id", "size"),
                Unique_ponds=("pond_id", "nunique"),
                Median_pH=("pH", "median"),
                Median_DO=("DO (mg/L)", "median"),
                Median_temperature=("Temp (in °C)", "median")
            )
            .reset_index()
        )

        summary["Region"] = region
        summary["Model"] = model_name

        condition_rows.append(summary)

# Consolidate regional and model-specific results.
period_condition_analysis = pd.concat(
    condition_rows,
    ignore_index=True
)

period_condition_analysis = (
    period_condition_analysis[
        [
            "Region",
            "Model",
            "Measurement period",
            "Detection outcome",
            "Observations",
            "Unique_ponds",
            "Median_pH",
            "Median_DO",
            "Median_temperature"
        ]
    ]
    .rename(columns={
        "Unique_ponds": "Unique ponds",
        "Median_pH": "Median pH",
        "Median_DO": "Median DO (mg/L)",
        "Median_temperature": "Median temperature (°C)"
    })
)

measurement_columns = [
    "Median pH",
    "Median DO (mg/L)",
    "Median temperature (°C)"
]

period_condition_analysis[measurement_columns] = (
    period_condition_analysis[measurement_columns].round(2)
)

display(period_condition_analysis)

,Region,Model,Measurement period,Detection outcome,Observations,Unique ponds,Median pH,Median DO (mg/L),Median temperature (°C)
0,Eluru,Logistic Regression,Later day,Detected (TP),50,23,8.76,11.35,31.60
1,Eluru,Logistic Regression,Later day,Missed (FN),105,57,8.41,11.30,29.30
2,Eluru,Logistic Regression,Morning,Detected (TP),141,56,8.38,3.00,29.10
3,Eluru,Logistic Regression,Morning,Missed (FN),16,9,7.98,2.90,25.30
4,Eluru,Random Forest,Later day,Detected (TP),61,34,8.53,11.80,31.30
5,Eluru,Random Forest,Later day,Missed (FN),94,56,8.42,11.20,30.10
6,Eluru,Random Forest,Morning,Detected (TP),102,47,8.46,2.40,29.15
7,Eluru,Random Forest,Morning,Missed (FN),55,31,8.22,3.30,28.60
8,Nellore,Logistic Regression,Later day,Detected (TP),3,1,8.89,8.70,33.90
9,Nellore,Logistic Regression,Later day,Missed (FN),19,16,8.31,9.10,32.80


In [12]:
# CELL 12: PREDICTED RISK SCORES FOR DETECTED AND MISSED CASES

score_data = error_analysis_data.copy()

# Recover the earlier observation's measurement period.
score_data["Observation timestamp"] = pd.to_datetime(
    score_data["Observation timestamp"],
    errors="coerce"
)

score_data["Measurement hour"] = (
    score_data["Observation timestamp"].dt.hour
)

score_data["Measurement period"] = np.select(
    [
        score_data["Measurement hour"].between(6, 8),
        score_data["Measurement hour"].between(15, 17)
    ],
    [
        "Morning",
        "Later day"
    ],
    default="Other hours"
)

model_configurations = {
    "Logistic Regression": {
        "prediction": "LR predicted class",
        "probability": "LR probability",
        "threshold": "LR threshold"
    },
    "Random Forest": {
        "prediction": "RF predicted class",
        "probability": "RF probability",
        "threshold": "RF threshold"
    }
}

score_rows = []

# Consider actual out-of-range observations only.
score_data = score_data[
    score_data["Actual class"] == 1
].copy()

for region in ["Eluru", "Nellore"]:

    region_data = score_data[
        score_data["Region"] == region
    ].copy()

    for model_name, config in model_configurations.items():

        prediction_column = config["prediction"]
        probability_column = config["probability"]
        threshold_column = config["threshold"]

        region_data["Detection outcome"] = np.where(
            region_data[prediction_column] == 1,
            "Detected (TP)",
            "Missed (FN)"
        )

        summary = (
            region_data
            .groupby(
                ["Measurement period", "Detection outcome"],
                observed=True
            )
            .agg(
                Observations=("pond_id", "size"),
                Unique_ponds=("pond_id", "nunique"),
                Mean_probability=(probability_column, "mean"),
                Median_probability=(probability_column, "median"),
                Probability_25th=(probability_column,
                                  lambda x: x.quantile(0.25)),
                Probability_75th=(probability_column,
                                  lambda x: x.quantile(0.75)),
                Threshold=(threshold_column, "first")
            )
            .reset_index()
        )

        summary["Region"] = region
        summary["Model"] = model_name

        score_rows.append(summary)

risk_score_analysis = pd.concat(
    score_rows,
    ignore_index=True
)

risk_score_analysis = risk_score_analysis[
    [
        "Region",
        "Model",
        "Measurement period",
        "Detection outcome",
        "Observations",
        "Unique_ponds",
        "Threshold",
        "Mean_probability",
        "Median_probability",
        "Probability_25th",
        "Probability_75th"
    ]
].rename(columns={
    "Unique_ponds": "Unique ponds",
    "Mean_probability": "Mean predicted probability",
    "Median_probability": "Median predicted probability",
    "Probability_25th": "25th percentile",
    "Probability_75th": "75th percentile"
})

numeric_columns = [
    "Threshold",
    "Mean predicted probability",
    "Median predicted probability",
    "25th percentile",
    "75th percentile"
]

risk_score_analysis[numeric_columns] = (
    risk_score_analysis[numeric_columns].round(4)
)

display(risk_score_analysis)

,Region,Model,Measurement period,Detection outcome,Observations,Unique ponds,Threshold,Mean predicted probability,Median predicted probability,25th percentile,75th percentile
0,Eluru,Logistic Regression,Later day,Detected (TP),50,23,0.2,0.2799,0.2638,0.2174,0.3293
1,Eluru,Logistic Regression,Later day,Missed (FN),105,57,0.2,0.1537,0.1578,0.1337,0.1760
2,Eluru,Logistic Regression,Morning,Detected (TP),141,56,0.2,0.3510,0.3275,0.2736,0.4200
3,Eluru,Logistic Regression,Morning,Missed (FN),16,9,0.2,0.1674,0.1656,0.1535,0.1875
4,Eluru,Random Forest,Later day,Detected (TP),61,34,0.4,0.5475,0.5323,0.4540,0.6428
5,Eluru,Random Forest,Later day,Missed (FN),94,56,0.4,0.2793,0.3005,0.2168,0.3331
6,Eluru,Random Forest,Morning,Detected (TP),102,47,0.4,0.8439,0.8927,0.7699,0.9426
7,Eluru,Random Forest,Morning,Missed (FN),55,31,0.4,0.2000,0.2005,0.1235,0.2599
8,Nellore,Logistic Regression,Later day,Detected (TP),3,1,0.4,0.4092,0.4085,0.4073,0.4108
9,Nellore,Logistic Regression,Later day,Missed (FN),19,16,0.4,0.1546,0.1536,0.1286,0.1830


In [13]:
# CELL 13: PREDICTION SCORE DISTANCE FROM THE ALERT THRESHOLD

margin_data = error_analysis_data[
    error_analysis_data["Actual class"] == 1
].copy()

margin_data["Observation timestamp"] = pd.to_datetime(
    margin_data["Observation timestamp"],
    errors="coerce"
)

margin_data["Measurement hour"] = (
    margin_data["Observation timestamp"].dt.hour
)

margin_data["Measurement period"] = np.select(
    [
        margin_data["Measurement hour"].between(6, 8),
        margin_data["Measurement hour"].between(15, 17)
    ],
    [
        "Morning",
        "Later day"
    ],
    default="Other hours"
)

model_configurations = {
    "Logistic Regression": {
        "prediction": "LR predicted class",
        "probability": "LR probability",
        "threshold": "LR threshold"
    },
    "Random Forest": {
        "prediction": "RF predicted class",
        "probability": "RF probability",
        "threshold": "RF threshold"
    }
}

margin_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = margin_data[
        margin_data["Region"] == region
    ].copy()

    for model_name, config in model_configurations.items():

        prediction_column = config["prediction"]
        probability_column = config["probability"]
        threshold_column = config["threshold"]

        for period in [
            "Morning",
            "Later day",
            "Other hours"
        ]:

            period_data = region_data[
                region_data["Measurement period"] == period
            ]

            for outcome, predicted_class in [
                ("Detected (TP)", 1),
                ("Missed (FN)", 0)
            ]:

                group = period_data[
                    period_data[prediction_column] == predicted_class
                ]

                if group.empty:
                    continue

                threshold = group[threshold_column].iloc[0]

                if predicted_class == 1:
                    margin = (
                        group[probability_column] - threshold
                    )
                else:
                    margin = (
                        threshold - group[probability_column]
                    )

                margin_rows.append({

                    "Region": region,
                    "Model": model_name,
                    "Measurement period": period,
                    "Detection outcome": outcome,
                    "Observations": len(group),
                    "Unique ponds": group["pond_id"].nunique(),
                    "Threshold": threshold,

                    "Mean distance": margin.mean(),
                    "Median distance": margin.median(),
                    "25th percentile": margin.quantile(0.25),
                    "75th percentile": margin.quantile(0.75),

                    "Within 0.05 of threshold (%)": (
                        margin.le(0.05).mean() * 100
                    ),

                    "Within 0.10 of threshold (%)": (
                        margin.le(0.10).mean() * 100
                    )
                })

threshold_margin_analysis = pd.DataFrame(margin_rows)

numeric_columns = [
    "Mean distance",
    "Median distance",
    "25th percentile",
    "75th percentile",
    "Within 0.05 of threshold (%)",
    "Within 0.10 of threshold (%)"
]

threshold_margin_analysis[numeric_columns] = (
    threshold_margin_analysis[numeric_columns].round(4)
)

display(threshold_margin_analysis)

,Region,Model,Measurement period,Detection outcome,Observations,Unique ponds,Threshold,Mean distance,Median distance,25th percentile,75th percentile,Within 0.05 of threshold (%),Within 0.10 of threshold (%)
0,Eluru,Logistic Regression,Morning,Detected (TP),141,56,0.2,0.1510,0.1275,0.0736,0.2200,17.7305,36.8794
1,Eluru,Logistic Regression,Morning,Missed (FN),16,9,0.2,0.0326,0.0344,0.0125,0.0465,75.0000,100.0000
2,Eluru,Logistic Regression,Later day,Detected (TP),50,23,0.2,0.0799,0.0638,0.0174,0.1293,44.0000,66.0000
3,Eluru,Logistic Regression,Later day,Missed (FN),105,57,0.2,0.0463,0.0422,0.0240,0.0663,63.8095,95.2381
4,Eluru,Random Forest,Morning,Detected (TP),102,47,0.4,0.4439,0.4927,0.3699,0.5426,0.9804,1.9608
5,Eluru,Random Forest,Morning,Missed (FN),55,31,0.4,0.2000,0.1995,0.1401,0.2765,7.2727,14.5455
6,Eluru,Random Forest,Later day,Detected (TP),61,34,0.4,0.1475,0.1323,0.0540,0.2428,24.5902,40.9836
7,Eluru,Random Forest,Later day,Missed (FN),94,56,0.4,0.1207,0.0995,0.0669,0.1832,17.0213,50.0000
8,Nellore,Logistic Regression,Morning,Detected (TP),14,11,0.4,0.1428,0.1616,0.0921,0.1826,7.1429,42.8571
9,Nellore,Logistic Regression,Morning,Missed (FN),8,8,0.4,0.1406,0.0922,0.0271,0.2445,37.5000,50.0000


In [14]:
# CELL 14: POND-LEVEL BOOTSTRAP — MORNING VS LATER-DAY RECALL

rng = np.random.default_rng(42)
n_bootstrap = 2000

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

period_recall_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = error_analysis_data[
        (error_analysis_data["Region"] == region)
        & (error_analysis_data["Actual class"] == 1)
    ].copy()

    # Derive measurement hour from the earlier observation timestamp.
    region_data["Observation timestamp"] = pd.to_datetime(
        region_data["Observation timestamp"],
        errors="coerce"
    )

    region_data["Measurement hour"] = (
        region_data["Observation timestamp"].dt.hour
    )

    region_data["Measurement period"] = np.select(
        [
            region_data["Measurement hour"].between(6, 8),
            region_data["Measurement hour"].between(15, 17)
        ],
        [
            "Morning",
            "Later day"
        ],
        default="Other hours"
    )

    # Retain the two main measurement periods.
    region_data = region_data[
        region_data["Measurement period"].isin(
            ["Morning", "Later day"]
        )
    ].copy()

    unique_ponds = region_data["pond_id"].unique()
    pond_ids = region_data["pond_id"].to_numpy()
    
    for model_name, prediction_column in model_columns.items():

        detected = (
            region_data[prediction_column].to_numpy() == 1
        )

        morning_mask = (
            region_data["Measurement period"].to_numpy() == "Morning"
        )

        later_mask = (
            region_data["Measurement period"].to_numpy() == "Later day"
        )

        morning_total = int(morning_mask.sum())
        later_total = int(later_mask.sum())

        morning_detected = int(
            (detected & morning_mask).sum()
        )

        later_detected = int(
            (detected & later_mask).sum()
        )

        morning_recall = morning_detected / morning_total
        later_recall = later_detected / later_total

        observed_difference = (
            later_recall - morning_recall
        ) * 100

        bootstrap_differences = []

        for _ in range(n_bootstrap):

            sampled_ponds = rng.choice(
                unique_ponds,
                size=len(unique_ponds),
                replace=True
            )

            sampled_positions = np.concatenate([
                np.flatnonzero(pond_ids == pond)
                for pond in sampled_ponds
            ])

            sampled_morning = morning_mask[sampled_positions]
            sampled_later = later_mask[sampled_positions]
            sampled_detected = detected[sampled_positions]

            morning_count = sampled_morning.sum()
            later_count = sampled_later.sum()

            if morning_count == 0 or later_count == 0:
                continue

            morning_boot_recall = (
                (sampled_detected & sampled_morning).sum()
                / morning_count
            )

            later_boot_recall = (
                (sampled_detected & sampled_later).sum()
                / later_count
            )

            bootstrap_differences.append(
                (later_boot_recall - morning_boot_recall) * 100
            )

        period_recall_rows.append({

            "Region": region,
            "Model": model_name,

            "Morning observations": morning_total,
            "Later-day observations": later_total,

            "Morning recall (%)": morning_recall * 100,
            "Later-day recall (%)": later_recall * 100,

            "Observed difference (percentage points)": (
                observed_difference
            ),

            "Bootstrap mean difference (percentage points)": (
                np.mean(bootstrap_differences)
            ),

            "95% CI lower": np.percentile(
                bootstrap_differences, 2.5
            ),

            "95% CI upper": np.percentile(
                bootstrap_differences, 97.5
            ),

            "Valid bootstrap samples": len(bootstrap_differences)

        })

period_recall_bootstrap = pd.DataFrame(period_recall_rows)

numeric_columns = [
    "Morning recall (%)",
    "Later-day recall (%)",
    "Observed difference (percentage points)",
    "Bootstrap mean difference (percentage points)",
    "95% CI lower",
    "95% CI upper"
]

period_recall_bootstrap[numeric_columns] = (
    period_recall_bootstrap[numeric_columns].round(2)
)

display(period_recall_bootstrap)

,Region,Model,Morning observations,Later-day observations,Morning recall (%),Later-day recall (%),Observed difference (percentage points),Bootstrap mean difference (percentage points),95% CI lower,95% CI upper,Valid bootstrap samples
0,Eluru,Logistic Regression,157,155,89.81,32.26,-57.55,-57.71,-69.61,-46.94,2000
1,Eluru,Random Forest,157,155,64.97,39.35,-25.61,-25.85,-37.58,-13.88,2000
2,Nellore,Logistic Regression,22,22,63.64,13.64,-50.00,-50.47,-70.59,-32.00,2000
3,Nellore,Random Forest,22,22,95.45,0.00,-95.45,-95.29,-100.00,-83.33,2000


In [ ]:
# CELL 17: CORRECTED VERSION OF CELL 16
# WITHIN-POND MORNING VS LATER-DAY RECALL

within_pond_data = error_analysis_data.copy()

within_pond_data["Observation timestamp"] = pd.to_datetime(
    within_pond_data["Observation timestamp"],
    errors="coerce"
)

within_pond_data["Measurement hour"] = (
    within_pond_data["Observation timestamp"].dt.hour
)

within_pond_data["Measurement period"] = np.select(
    [
        within_pond_data["Measurement hour"].between(6, 8),
        within_pond_data["Measurement hour"].between(15, 17)
    ],
    ["Morning", "Later day"],
    default="Other hours"
)

# Retain actual out-of-range observations from the two periods.
within_pond_data = within_pond_data[
    (within_pond_data["Actual class"] == 1)
    & within_pond_data["Measurement period"].isin(
        ["Morning", "Later day"]
    )
].copy()

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

within_pond_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = within_pond_data[
        within_pond_data["Region"] == region
    ].copy()

    for model_name, prediction_column in model_columns.items():

        # Calculate detection counts for each pond and period.
        pond_period = (
            region_data
            .groupby(["pond_id", "Measurement period"])
            .agg(
                actual_out_of_range=("Actual class", "size"),
                detected=(prediction_column, "sum")
            )
            .reset_index()
        )

        pond_period["recall"] = (
            pond_period["detected"]
            / pond_period["actual_out_of_range"]
        )

        # Arrange morning and later-day recall side by side.
        recall_wide = pond_period.pivot(
            index="pond_id",
            columns="Measurement period",
            values="recall"
        )

        # Include only ponds having observations in both periods.
        if {"Morning", "Later day"}.issubset(recall_wide.columns):

            paired_ponds = recall_wide.dropna(
                subset=["Morning", "Later day"]
            ).copy()

        else:
            paired_ponds = pd.DataFrame(
                columns=["Morning", "Later day"]
            )

        paired_ponds["recall_difference"] = (
            paired_ponds["Later day"]
            - paired_ponds["Morning"]
        )

        differences = paired_ponds["recall_difference"]

        within_pond_rows.append({

            "Region": region,
            "Model": model_name,

            "Ponds with both periods": len(paired_ponds),

            "Median morning recall (%)": (
                paired_ponds["Morning"].median() * 100
            ),

            "Median later-day recall (%)": (
                paired_ponds["Later day"].median() * 100
            ),

            "Median within-pond difference (pp)": (
                differences.median() * 100
            ),

            "Ponds with lower later-day recall": (
                (differences < 0).sum()
            ),

            "Ponds with higher later-day recall": (
                (differences > 0).sum()
            ),

            "Ponds with equal recall": (
                (differences == 0).sum()
            )
        })

# Create one consolidated output.
within_pond_comparison = pd.DataFrame(within_pond_rows)

numeric_columns = [
    "Median morning recall (%)",
    "Median later-day recall (%)",
    "Median within-pond difference (pp)"
]

within_pond_comparison[numeric_columns] = (
    within_pond_comparison[numeric_columns].round(2)
)

display(within_pond_comparison)

,Region,Model,Ponds with both periods,Median morning recall (%),Median later-day recall (%),Median within-pond difference (pp),Ponds with lower later-day recall,Ponds with higher later-day recall,Ponds with equal recall
0,Eluru,Logistic Regression,51,100.0,0.0,-66.67,43,0,8
1,Eluru,Random Forest,51,100.0,50.0,-45.71,36,10,5
2,Nellore,Logistic Regression,14,75.0,0.0,-37.50,8,0,6
3,Nellore,Random Forest,14,100.0,0.0,-100.00,13,0,1


In [ ]:
# CELL 18: PAIRED POND-LEVEL BOOTSTRAP FOR MEASUREMENT-PERIOD RECALL

rng = np.random.default_rng(42)
n_bootstrap = 10000

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

paired_recall_results = []

# Consider actual out-of-range observations only.
positive_data = error_analysis_data[
    error_analysis_data["Actual class"] == 1
].copy()

positive_data["Observation timestamp"] = pd.to_datetime(
    positive_data["Observation timestamp"],
    errors="coerce"
)

positive_data["Measurement hour"] = (
    positive_data["Observation timestamp"].dt.hour
)

positive_data["Measurement period"] = np.select(
    [
        positive_data["Measurement hour"].between(6, 8),
        positive_data["Measurement hour"].between(15, 17)
    ],
    ["Morning", "Later day"],
    default="Other hours"
)

positive_data = positive_data[
    positive_data["Measurement period"].isin(
        ["Morning", "Later day"]
    )
].copy()

for region in ["Eluru", "Nellore"]:

    region_data = positive_data[
        positive_data["Region"] == region
    ].copy()

    for model_name, prediction_column in model_columns.items():

        # Calculate recall separately for each pond and period.
        pond_period = (
            region_data
            .groupby(["pond_id", "Measurement period"])
            .agg(
                actual_cases=("Actual class", "size"),
                detected=(prediction_column, "sum")
            )
            .reset_index()
        )

        pond_period["Recall"] = (
            pond_period["detected"]
            / pond_period["actual_cases"]
        )

        # Pair morning and later-day recall within each pond.
        paired = pond_period.pivot(
            index="pond_id",
            columns="Measurement period",
            values="Recall"
        ).dropna(
            subset=["Morning", "Later day"]
        )

        if len(paired) == 0:
            continue

        paired["Difference"] = (
            paired["Later day"] - paired["Morning"]
        )

        differences = paired["Difference"].to_numpy()

        # Paired bootstrap: resample whole pond-level differences.
        bootstrap_means = []

        for _ in range(n_bootstrap):

            sampled = rng.choice(
                differences,
                size=len(differences),
                replace=True
            )

            bootstrap_means.append(sampled.mean() * 100)

        paired_recall_results.append({

            "Region": region,
            "Model": model_name,

            "Ponds compared": len(paired),

            "Mean paired difference (pp)": (
                differences.mean() * 100
            ),

            "Median paired difference (pp)": (
                np.median(differences) * 100
            ),

            "95% CI lower": np.percentile(
                bootstrap_means, 2.5
            ),

            "95% CI upper": np.percentile(
                bootstrap_means, 97.5
            ),

            "Ponds with lower later-day recall": (
                differences < 0
            ).sum(),

            "Ponds with higher later-day recall": (
                differences > 0
            ).sum(),

            "Ponds with equal recall": (
                differences == 0
            ).sum()

        })

paired_recall_bootstrap = pd.DataFrame(
    paired_recall_results
)

numeric_columns = [
    "Mean paired difference (pp)",
    "Median paired difference (pp)",
    "95% CI lower",
    "95% CI upper"
]

paired_recall_bootstrap[numeric_columns] = (
    paired_recall_bootstrap[numeric_columns].round(2)
)

display(paired_recall_bootstrap)

,Region,Model,Ponds compared,Mean paired difference (pp),Median paired difference (pp),95% CI lower,95% CI upper,Ponds with lower later-day recall,Ponds with higher later-day recall,Ponds with equal recall
0,Eluru,Logistic Regression,51,-63.05,-66.67,-73.51,-52.03,43,0,8
1,Eluru,Random Forest,51,-36.80,-45.71,-51.34,-21.72,36,10,5
2,Nellore,Logistic Regression,14,-48.21,-37.50,-71.43,-25.00,8,0,6
3,Nellore,Random Forest,14,-92.86,-100.00,-100.00,-78.57,13,0,1


In [ ]:
# CELL 19: WITHIN-POND WATER-QUALITY COMPARISON BY MEASUREMENT PERIOD

# Select actual out-of-range prediction cases.
paired_condition_data = error_analysis_data[
    error_analysis_data["Actual class"] == 1
].copy()

# Recover earlier observation measurement time.
paired_condition_data["Observation timestamp"] = pd.to_datetime(
    paired_condition_data["Observation timestamp"],
    errors="coerce"
)

paired_condition_data["Measurement hour"] = (
    paired_condition_data["Observation timestamp"].dt.hour
)

paired_condition_data["Measurement period"] = np.select(
    [
        paired_condition_data["Measurement hour"].between(6, 8),
        paired_condition_data["Measurement hour"].between(15, 17)
    ],
    ["Morning", "Later day"],
    default="Other hours"
)

# Retain morning and later-day observations only.
paired_condition_data = paired_condition_data[
    paired_condition_data["Measurement period"].isin(
        ["Morning", "Later day"]
    )
].copy()

predictors = [
    "pH",
    "DO (mg/L)",
    "Temp (in °C)"
]

paired_condition_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = paired_condition_data[
        paired_condition_data["Region"] == region
    ].copy()

    for feature in predictors:

        # Calculate the median measurement for each pond and period.
        pond_measurements = (
            region_data
            .groupby(["pond_id", "Measurement period"])[feature]
            .median()
            .unstack("Measurement period")
        )

        # Retain ponds with measurements in both periods.
        paired = pond_measurements.dropna(
            subset=["Morning", "Later day"]
        ).copy()

        # Calculate within-pond changes.
        paired["Difference"] = (
            paired["Later day"] - paired["Morning"]
        )

        differences = paired["Difference"]

        paired_condition_rows.append({

            "Region": region,
            "Parameter": feature,

            "Ponds compared": len(paired),

            "Median morning value": paired["Morning"].median(),

            "Median later-day value": paired["Later day"].median(),

            "Median paired difference": differences.median(),

            "25th percentile difference": differences.quantile(0.25),

            "75th percentile difference": differences.quantile(0.75),

            "Ponds with increase": (differences > 0).sum(),

            "Ponds with decrease": (differences < 0).sum(),

            "Ponds with no change": (differences == 0).sum()

        })

# Consolidate results.
within_pond_condition_comparison = pd.DataFrame(
    paired_condition_rows
)

numeric_columns = [
    "Median morning value",
    "Median later-day value",
    "Median paired difference",
    "25th percentile difference",
    "75th percentile difference"
]

within_pond_condition_comparison[numeric_columns] = (
    within_pond_condition_comparison[numeric_columns].round(3)
)

display(within_pond_condition_comparison)

,Region,Parameter,Ponds compared,Median morning value,Median later-day value,Median paired difference,25th percentile difference,75th percentile difference,Ponds with increase,Ponds with decrease,Ponds with no change
0,Eluru,pH,51,8.380,8.45,0.130,-0.008,0.240,37,13,1
1,Eluru,DO (mg/L),51,2.450,11.50,8.900,7.975,9.900,51,0,0
2,Eluru,Temp (in °C),51,28.750,30.60,2.100,1.200,2.950,46,5,0
3,Nellore,pH,14,8.240,8.33,0.190,0.002,0.347,10,3,1
4,Nellore,DO (mg/L),14,2.600,9.20,6.700,5.400,7.750,14,0,0
5,Nellore,Temp (in °C),14,29.925,32.60,2.675,1.925,3.500,14,0,0


In [ ]:
# CELL 20: WILCOXON TEST — WITHIN-POND MORNING VS LATER-DAY DIFFERENCES

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

predictors = [
    "pH",
    "DO (mg/L)",
    "Temp (in °C)"
]

statistical_test_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = paired_condition_data[
        paired_condition_data["Region"] == region
    ].copy()

    for feature in predictors:

        # Calculate pond-level median measurements in each period.
        pond_medians = (
            region_data
            .groupby(["pond_id", "Measurement period"])[feature]
            .median()
            .unstack("Measurement period")
        )

        paired = pond_medians.dropna(
            subset=["Morning", "Later day"]
        )

        morning = paired["Morning"]
        later_day = paired["Later day"]

        differences = later_day - morning

        # Paired Wilcoxon signed-rank test.
        test_result = wilcoxon(
            morning,
            later_day,
            alternative="two-sided",
            zero_method="wilcox",
            method="auto"
        )

        statistical_test_rows.append({

            "Region": region,
            "Parameter": feature,

            "Ponds compared": len(paired),

            "Median morning": morning.median(),

            "Median later day": later_day.median(),

            "Median paired difference": differences.median(),

            "Ponds with increase": (differences > 0).sum(),

            "Ponds with decrease": (differences < 0).sum(),

            "Ponds with no change": (differences == 0).sum(),

            "Wilcoxon statistic": test_result.statistic,

            "Raw p-value": test_result.pvalue

        })

# Create one consolidated statistical table.
within_pond_statistical_tests = pd.DataFrame(
    statistical_test_rows
)

# Adjust for the six comparisons using Holm's method.
adjusted_results = multipletests(
    within_pond_statistical_tests["Raw p-value"],
    method="holm"
)

within_pond_statistical_tests["Holm-adjusted p-value"] = (
    adjusted_results[1]
)

within_pond_statistical_tests["Significant at 0.05"] = (
    within_pond_statistical_tests["Holm-adjusted p-value"] < 0.05
)

numeric_columns = [
    "Median morning",
    "Median later day",
    "Median paired difference",
    "Wilcoxon statistic",
    "Raw p-value",
    "Holm-adjusted p-value"
]

within_pond_statistical_tests[numeric_columns] = (
    within_pond_statistical_tests[numeric_columns].round(5)
)

display(within_pond_statistical_tests)

,Region,Parameter,Ponds compared,Median morning,Median later day,Median paired difference,Ponds with increase,Ponds with decrease,Ponds with no change,Wilcoxon statistic,Raw p-value,Holm-adjusted p-value,Significant at 0.05
0,Eluru,pH,51,8.380,8.45,0.130,37,13,1,341.5,0.00427,0.00854,True
1,Eluru,DO (mg/L),51,2.450,11.50,8.900,51,0,0,0.0,0.00000,0.00000,True
2,Eluru,Temp (in °C),51,28.750,30.60,2.100,46,5,0,30.0,0.00000,0.00000,True
3,Nellore,pH,14,8.240,8.33,0.190,10,3,1,14.0,0.02771,0.02771,True
4,Nellore,DO (mg/L),14,2.600,9.20,6.700,14,0,0,0.0,0.00012,0.00049,True
5,Nellore,Temp (in °C),14,29.925,32.60,2.675,14,0,0,0.0,0.00098,0.00293,True


In [ ]:
# CELL 21: WITHIN-POND MEASUREMENT DIFFERENCES
# ACROSS ALL TEST OBSERVATIONS

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

all_test_data = error_analysis_data.copy()

all_test_data["Observation timestamp"] = pd.to_datetime(
    all_test_data["Observation timestamp"],
    errors="coerce"
)

all_test_data["Measurement hour"] = (
    all_test_data["Observation timestamp"].dt.hour
)

all_test_data["Measurement period"] = np.select(
    [
        all_test_data["Measurement hour"].between(6, 8),
        all_test_data["Measurement hour"].between(15, 17)
    ],
    ["Morning", "Later day"],
    default="Other hours"
)

all_test_data = all_test_data[
    all_test_data["Measurement period"].isin(
        ["Morning", "Later day"]
    )
].copy()

predictors = [
    "pH",
    "DO (mg/L)",
    "Temp (in °C)"
]

all_observation_results = []

for region in ["Eluru", "Nellore"]:

    region_data = all_test_data[
        all_test_data["Region"] == region
    ].copy()

    for feature in predictors:

        # Calculate pond-level median values in each period.
        pond_measurements = (
            region_data
            .groupby(["pond_id", "Measurement period"])[feature]
            .median()
            .unstack("Measurement period")
        )

        # Keep ponds represented in both measurement periods.
        paired = pond_measurements.dropna(
            subset=["Morning", "Later day"]
        )

        morning = paired["Morning"]
        later_day = paired["Later day"]

        differences = later_day - morning

        test_result = wilcoxon(
            morning,
            later_day,
            alternative="two-sided",
            zero_method="wilcox",
            method="auto"
        )

        all_observation_results.append({

            "Region": region,
            "Parameter": feature,

            "Ponds compared": len(paired),

            "Median morning": morning.median(),
            "Median later day": later_day.median(),

            "Median paired difference": differences.median(),

            "Ponds with increase": (differences > 0).sum(),
            "Ponds with decrease": (differences < 0).sum(),
            "Ponds with no change": (differences == 0).sum(),

            "Raw p-value": test_result.pvalue

        })

all_test_period_analysis = pd.DataFrame(
    all_observation_results
)

# Correct for six statistical comparisons.
all_test_period_analysis["Holm-adjusted p-value"] = (
    multipletests(
        all_test_period_analysis["Raw p-value"],
        method="holm"
    )[1]
)

all_test_period_analysis["Significant at 0.05"] = (
    all_test_period_analysis["Holm-adjusted p-value"] < 0.05
)

numeric_columns = [
    "Median morning",
    "Median later day",
    "Median paired difference",
    "Raw p-value",
    "Holm-adjusted p-value"
]

all_test_period_analysis[numeric_columns] = (
    all_test_period_analysis[numeric_columns].round(5)
)

display(all_test_period_analysis)

,Region,Parameter,Ponds compared,Median morning,Median later day,Median paired difference,Ponds with increase,Ponds with decrease,Ponds with no change,Raw p-value,Holm-adjusted p-value,Significant at 0.05
0,Eluru,pH,113,8.18,8.42,0.215,113,0,0,0.0,0.0,True
1,Eluru,DO (mg/L),113,3.30,11.10,7.650,113,0,0,0.0,0.0,True
2,Eluru,Temp (in °C),113,28.50,30.30,1.950,104,8,1,0.0,0.0,True
3,Nellore,pH,47,8.10,8.31,0.220,47,0,0,0.0,0.0,True
4,Nellore,DO (mg/L),47,3.70,9.50,5.800,47,0,0,0.0,0.0,True
5,Nellore,Temp (in °C),47,30.20,32.85,2.850,47,0,0,0.0,0.0,True


In [ ]:
# CELL 22: EXACT P-VALUES FOR ALL-OBSERVATION TIME-OF-DAY COMPARISON

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

exact_test_rows = []

predictors = ["pH", "DO (mg/L)", "Temp (in °C)"]

for region in ["Eluru", "Nellore"]:

    region_data = all_test_data[
        all_test_data["Region"] == region
    ]

    for feature in predictors:

        pond_measurements = (
            region_data
            .groupby(["pond_id", "Measurement period"])[feature]
            .median()
            .unstack("Measurement period")
        )

        paired = pond_measurements.dropna(
            subset=["Morning", "Later day"]
        )

        differences = (
            paired["Later day"] - paired["Morning"]
        )

        result = wilcoxon(
            paired["Morning"],
            paired["Later day"],
            alternative="two-sided",
            zero_method="wilcox",
            method="auto"
        )

        exact_test_rows.append({
            "Region": region,
            "Parameter": feature,
            "Ponds compared": len(paired),
            "Wilcoxon statistic": result.statistic,
            "Raw p-value": result.pvalue
        })

exact_time_tests = pd.DataFrame(exact_test_rows)

exact_time_tests["Holm-adjusted p-value"] = multipletests(
    exact_time_tests["Raw p-value"],
    method="holm"
)[1]

exact_time_tests["Significant at 0.05"] = (
    exact_time_tests["Holm-adjusted p-value"] < 0.05
)

# Display p-values in scientific notation.
exact_time_display = exact_time_tests.copy()

for column in ["Raw p-value", "Holm-adjusted p-value"]:
    exact_time_display[column] = (
        exact_time_display[column].map(lambda x: f"{x:.3e}")
    )

display(exact_time_display)

,Region,Parameter,Ponds compared,Wilcoxon statistic,Raw p-value,Holm-adjusted p-value,Significant at 0.05
0,Eluru,pH,113,0.0,2.789e-20,1.668e-19,True
1,Eluru,DO (mg/L),113,0.0,2.780e-20,1.668e-19,True
2,Eluru,Temp (in °C),113,108.5,7.258e-19,2.903e-18,True
3,Nellore,pH,47,0.0,2.382e-09,7.145e-09,True
4,Nellore,DO (mg/L),47,0.0,2.384e-09,7.145e-09,True
5,Nellore,Temp (in °C),47,0.0,2.389e-09,7.145e-09,True


In [ ]:
# CELL 23: MONTHLY WITHIN-POND MEASUREMENT-PERIOD ANALYSIS

predictors = [
    "pH",
    "DO (mg/L)",
    "Temp (in °C)"
]

monthly_data = all_test_data.copy()

monthly_data["Observation timestamp"] = pd.to_datetime(
    monthly_data["Observation timestamp"],
    errors="coerce"
)

# Identify the calendar month of each earlier observation.
monthly_data["Calendar month"] = (
    monthly_data["Observation timestamp"]
    .dt.strftime("%Y-%m")
)

# Retain morning and later-day measurements.
monthly_data = monthly_data[
    monthly_data["Measurement period"].isin(
        ["Morning", "Later day"]
    )
].copy()

# Calculate pond-level monthly median measurements.
monthly_pond_summary = (
    monthly_data
    .groupby(
        [
            "Region",
            "pond_id",
            "Calendar month",
            "Measurement period"
        ]
    )[predictors]
    .median()
    .reset_index()
)

monthly_difference_rows = []

# Compare measurements within each pond and calendar month.
for feature in predictors:

    monthly_wide = monthly_pond_summary.pivot(
        index=["Region", "pond_id", "Calendar month"],
        columns="Measurement period",
        values=feature
    )

    # Retain pond-month combinations with both periods available.
    paired = monthly_wide.dropna(
        subset=["Morning", "Later day"]
    )

    difference = (
        paired["Later day"] - paired["Morning"]
    )

    paired_results = pd.DataFrame({

        "Region": paired.index.get_level_values("Region"),

        "Calendar month": paired.index.get_level_values(
            "Calendar month"
        ),

        "Parameter": feature,

        "Paired pond-months": 1,

        "Within-pond difference": difference.to_numpy(),

        "Ponds with increase": (
            difference.to_numpy() > 0
        ).astype(int),

        "Ponds with decrease": (
            difference.to_numpy() < 0
        ).astype(int),

        "Ponds with no change": (
            difference.to_numpy() == 0
        ).astype(int)

    })

    monthly_difference_rows.append(paired_results)

# Combine all parameter comparisons.
monthly_pond_differences = pd.concat(
    monthly_difference_rows,
    ignore_index=True
)

# Summarise differences by region, month and parameter.
monthly_measurement_analysis = (
    monthly_pond_differences
    .groupby(
        ["Region", "Calendar month", "Parameter"]
    )
    .agg(
        Paired_pond_months=("Paired pond-months", "sum"),

        Median_difference=(
            "Within-pond difference", "median"
        ),

        Difference_25th_percentile=(
            "Within-pond difference",
            lambda x: x.quantile(0.25)
        ),

        Difference_75th_percentile=(
            "Within-pond difference",
            lambda x: x.quantile(0.75)
        ),

        Ponds_with_increase=("Ponds with increase", "sum"),

        Ponds_with_decrease=("Ponds with decrease", "sum"),

        Ponds_with_no_change=("Ponds with no change", "sum")
    )
    .reset_index()
    .sort_values(
        ["Region", "Calendar month", "Parameter"]
    )
)

# Round numeric summaries.
numeric_columns = [
    "Median_difference",
    "Difference_25th_percentile",
    "Difference_75th_percentile"
]

monthly_measurement_analysis[numeric_columns] = (
    monthly_measurement_analysis[numeric_columns].round(3)
)

monthly_measurement_analysis = (
    monthly_measurement_analysis.rename(columns={
        "Paired_pond_months": "Paired pond-months",
        "Median_difference": "Median difference",
        "Difference_25th_percentile": "25th percentile",
        "Difference_75th_percentile": "75th percentile",
        "Ponds_with_increase": "Ponds with increase",
        "Ponds_with_decrease": "Ponds with decrease",
        "Ponds_with_no_change": "Ponds with no change"
    })
)

display(monthly_measurement_analysis)

,Region,Calendar month,Parameter,Paired pond-months,Median difference,25th percentile,75th percentile,Ponds with increase,Ponds with decrease,Ponds with no change
0,Eluru,2025-11,DO (mg/L),84,7.400,5.800,7.800,84,0,0
1,Eluru,2025-11,Temp (in °C),84,2.350,1.975,3.000,84,0,0
2,Eluru,2025-11,pH,84,0.305,0.124,0.432,84,0,0
3,Eluru,2025-12,DO (mg/L),70,7.000,6.400,7.500,70,0,0
4,Eluru,2025-12,Temp (in °C),70,2.000,1.750,2.375,70,0,0
5,Eluru,2025-12,pH,70,0.252,0.121,0.384,70,0,0
6,Eluru,2026-01,DO (mg/L),70,7.300,6.325,7.987,70,0,0
7,Eluru,2026-01,Temp (in °C),70,2.550,2.100,2.975,70,0,0
8,Eluru,2026-01,pH,70,0.295,0.140,0.418,69,1,0
9,Eluru,2026-02,DO (mg/L),80,7.875,7.500,8.400,80,0,0


In [ ]:
# CELL 24: POND-LEVEL BOOTSTRAP OF MONTHLY PAIRED DIFFERENCES

rng = np.random.default_rng(42)
n_bootstrap = 2000

predictors = [
    "pH",
    "DO (mg/L)",
    "Temp (in °C)"
]

clustered_difference_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = monthly_pond_summary[
        monthly_pond_summary["Region"] == region
    ].copy()

    for feature in predictors:

        # Pair morning and later-day values within the same pond-month.
        wide = region_data.pivot(
            index=["pond_id", "Calendar month"],
            columns="Measurement period",
            values=feature
        )

        paired = wide.dropna(
            subset=["Morning", "Later day"]
        ).copy()

        paired["Difference"] = (
            paired["Later day"] - paired["Morning"]
        )

        # Give each pond equal weight, regardless of its number of months.
        pond_mean_differences = (
            paired["Difference"]
            .groupby(level="pond_id")
            .mean()
        )

        differences = pond_mean_differences.to_numpy()

        if len(differences) < 2:
            continue

        # Bootstrap ponds rather than pond-month observations.
        bootstrap_means = []

        for _ in range(n_bootstrap):

            sampled = rng.choice(
                differences,
                size=len(differences),
                replace=True
            )

            bootstrap_means.append(sampled.mean())

        clustered_difference_rows.append({

            "Region": region,
            "Parameter": feature,

            "Paired pond-months": len(paired),
            "Unique ponds": len(differences),

            "Mean pond-level difference": differences.mean(),
            "Median pond-level difference": np.median(differences),

            "Ponds with positive difference": (
                differences > 0
            ).sum(),

            "Ponds with negative difference": (
                differences < 0
            ).sum(),

            "Ponds with zero difference": (
                differences == 0
            ).sum(),

            "Bootstrap 95% CI lower": np.percentile(
                bootstrap_means, 2.5
            ),

            "Bootstrap 95% CI upper": np.percentile(
                bootstrap_means, 97.5
            )

        })

clustered_measurement_results = pd.DataFrame(
    clustered_difference_rows
)

numeric_columns = [
    "Mean pond-level difference",
    "Median pond-level difference",
    "Bootstrap 95% CI lower",
    "Bootstrap 95% CI upper"
]

clustered_measurement_results[numeric_columns] = (
    clustered_measurement_results[numeric_columns].round(3)
)

display(clustered_measurement_results)

,Region,Parameter,Paired pond-months,Unique ponds,Mean pond-level difference,Median pond-level difference,Ponds with positive difference,Ponds with negative difference,Ponds with zero difference,Bootstrap 95% CI lower,Bootstrap 95% CI upper
0,Eluru,pH,615,113,0.251,0.255,113,0,0,0.234,0.268
1,Eluru,DO (mg/L),615,113,7.579,7.500,113,0,0,7.465,7.700
2,Eluru,Temp (in °C),615,113,2.390,2.279,113,0,0,2.304,2.476
3,Nellore,pH,225,47,0.236,0.233,47,0,0,0.227,0.245
4,Nellore,DO (mg/L),225,47,5.899,5.908,47,0,0,5.794,6.006
5,Nellore,Temp (in °C),225,47,2.710,2.612,47,0,0,2.564,2.852


In [25]:
# CELL 25: WITHIN-POND-MONTH COMPARISON OF MODEL RECALL

rng = np.random.default_rng(42)
n_bootstrap = 2000

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

matched_data = error_analysis_data[
    error_analysis_data["Actual class"] == 1
].copy()

matched_data["Observation timestamp"] = pd.to_datetime(
    matched_data["Observation timestamp"],
    errors="coerce"
)

matched_data["Measurement hour"] = (
    matched_data["Observation timestamp"].dt.hour
)

matched_data["Calendar month"] = (
    matched_data["Observation timestamp"].dt.strftime("%Y-%m")
)

matched_data["Measurement period"] = np.select(
    [
        matched_data["Measurement hour"].between(6, 8),
        matched_data["Measurement hour"].between(15, 17)
    ],
    ["Morning", "Later day"],
    default="Other hours"
)

matched_data = matched_data[
    matched_data["Measurement period"].isin(
        ["Morning", "Later day"]
    )
].copy()

matched_recall_results = []

for region in ["Eluru", "Nellore"]:

    region_data = matched_data[
        matched_data["Region"] == region
    ].copy()

    for model_name, prediction_column in model_columns.items():

        # Calculate recall for each pond, month and measurement period.
        grouped = (
            region_data
            .groupby(
                ["pond_id", "Calendar month", "Measurement period"]
            )
            .agg(
                actual_cases=("Actual class", "size"),
                detected=(prediction_column, "sum")
            )
            .reset_index()
        )

        grouped["Recall"] = (
            grouped["detected"] / grouped["actual_cases"]
        )

        # Match morning and later-day recall within pond-months.
        paired = grouped.pivot(
            index=["pond_id", "Calendar month"],
            columns="Measurement period",
            values="Recall"
        )

        if not {"Morning", "Later day"}.issubset(paired.columns):
            continue

        paired = paired.dropna(
            subset=["Morning", "Later day"]
        ).copy()

        if paired.empty:
            continue

        paired["Difference"] = (
            paired["Later day"] - paired["Morning"]
        )

        # Average across months within each pond.
        # Each pond receives equal weight in the final comparison.
        pond_means = (
            paired.groupby(level="pond_id")[
                ["Morning", "Later day", "Difference"]
            ].mean()
        )

        unique_ponds = len(pond_means)

        if unique_ponds < 2:
            continue

        differences = pond_means["Difference"].to_numpy()

        bootstrap_differences = []

        # Resample whole ponds.
        for _ in range(n_bootstrap):

            sampled = rng.choice(
                differences,
                size=unique_ponds,
                replace=True
            )

            bootstrap_differences.append(
                sampled.mean() * 100
            )

        matched_recall_results.append({

            "Region": region,
            "Model": model_name,

            "Matched pond-months": len(paired),
            "Unique ponds": unique_ponds,

            "Mean morning recall (%)": (
                pond_means["Morning"].mean() * 100
            ),

            "Mean later-day recall (%)": (
                pond_means["Later day"].mean() * 100
            ),

            "Mean paired difference (pp)": (
                differences.mean() * 100
            ),

            "95% CI lower": np.percentile(
                bootstrap_differences, 2.5
            ),

            "95% CI upper": np.percentile(
                bootstrap_differences, 97.5
            ),

            "Ponds with lower later-day recall": (
                differences < 0
            ).sum(),

            "Ponds with higher later-day recall": (
                differences > 0
            ).sum(),

            "Ponds with equal recall": (
                differences == 0
            ).sum()

        })

matched_month_recall_analysis = pd.DataFrame(
    matched_recall_results
)

numeric_columns = [
    "Mean morning recall (%)",
    "Mean later-day recall (%)",
    "Mean paired difference (pp)",
    "95% CI lower",
    "95% CI upper"
]

matched_month_recall_analysis[numeric_columns] = (
    matched_month_recall_analysis[numeric_columns].round(2)
)

display(matched_month_recall_analysis)

,Region,Model,Matched pond-months,Unique ponds,Mean morning recall (%),Mean later-day recall (%),Mean paired difference (pp),95% CI lower,95% CI upper,Ponds with lower later-day recall,Ponds with higher later-day recall,Ponds with equal recall
0,Eluru,Logistic Regression,48,34,85.62,33.33,-52.29,-65.52,-38.15,24,0,10
1,Eluru,Random Forest,48,34,77.16,42.73,-34.43,-49.76,-18.28,22,4,8
2,Nellore,Logistic Regression,4,3,66.67,33.33,-33.33,-100.00,0.00,1,0,2
3,Nellore,Random Forest,4,3,100.00,0.00,-100.00,-100.00,-100.00,3,0,0


In [26]:
# CELL 26: AUDIT MATCHED POND-MONTH COVERAGE

matched_coverage_rows = []

# Use actual out-of-range observations.
matched_coverage_data = all_test_data[
    (all_test_data["Actual class"] == 1)
    & all_test_data["Measurement period"].isin(
        ["Morning", "Later day"]
    )
].copy()

matched_coverage_data["Calendar month"] = (
    matched_coverage_data["Observation timestamp"]
    .dt.strftime("%Y-%m")
)

for region in ["Eluru", "Nellore"]:

    region_data = matched_coverage_data[
        matched_coverage_data["Region"] == region
    ]

    # Count observations within each pond-month-period.
    pond_month_counts = (
        region_data
        .groupby(
            ["pond_id", "Calendar month", "Measurement period"]
        )
        .size()
        .unstack("Measurement period")
    )

    # Retain pond-months having both measurement periods.
    matched = pond_month_counts.dropna(
        subset=["Morning", "Later day"]
    ).copy()

    # Count matched months contributed by each pond.
    pond_contributions = (
        matched.groupby(level="pond_id").size()
    )

    matched_coverage_rows.append({

        "Region": region,

        "Matched pond-months": len(matched),

        "Unique ponds": len(pond_contributions),

        "Minimum matched months per pond": (
            pond_contributions.min()
        ),

        "Median matched months per pond": (
            pond_contributions.median()
        ),

        "Maximum matched months per pond": (
            pond_contributions.max()
        ),

        "Ponds with one matched month": (
            (pond_contributions == 1).sum()
        ),

        "Ponds with multiple matched months": (
            (pond_contributions > 1).sum()
        )

    })

matched_coverage_audit = pd.DataFrame(
    matched_coverage_rows
)

display(matched_coverage_audit)

,Region,Matched pond-months,Unique ponds,Minimum matched months per pond,Median matched months per pond,Maximum matched months per pond,Ponds with one matched month,Ponds with multiple matched months
0,Eluru,48,34,1,1.0,3,24,10
1,Nellore,4,3,1,1.0,2,2,1


In [27]:
# CELL 27: COMPLETE CLASSIFICATION PERFORMANCE BY MEASUREMENT PERIOD

from sklearn.metrics import confusion_matrix

period_performance_data = error_analysis_data.copy()

period_performance_data["Observation timestamp"] = pd.to_datetime(
    period_performance_data["Observation timestamp"],
    errors="coerce"
)

period_performance_data["Measurement hour"] = (
    period_performance_data["Observation timestamp"].dt.hour
)

period_performance_data["Measurement period"] = np.select(
    [
        period_performance_data["Measurement hour"].between(6, 8),
        period_performance_data["Measurement hour"].between(15, 17)
    ],
    ["Morning", "Later day"],
    default="Other hours"
)

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

performance_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = period_performance_data[
        period_performance_data["Region"] == region
    ].copy()

    for model_name, prediction_column in model_columns.items():

        for period in [
            "Morning",
            "Later day",
            "Other hours"
        ]:

            group = region_data[
                region_data["Measurement period"] == period
            ]

            if group.empty:
                continue

            actual = group["Actual class"].astype(int)
            predicted = group[prediction_column].astype(int)

            tn, fp, fn, tp = confusion_matrix(
                actual,
                predicted,
                labels=[0, 1]
            ).ravel()

            precision = (
                tp / (tp + fp)
                if tp + fp else np.nan
            )

            recall = (
                tp / (tp + fn)
                if tp + fn else np.nan
            )

            specificity = (
                tn / (tn + fp)
                if tn + fp else np.nan
            )

            false_positive_rate = (
                fp / (fp + tn)
                if fp + tn else np.nan
            )

            f1 = (
                2 * precision * recall / (precision + recall)
                if (
                    pd.notna(precision)
                    and pd.notna(recall)
                    and precision + recall > 0
                )
                else np.nan
            )

            balanced_accuracy = (
                (recall + specificity) / 2
                if pd.notna(recall) and pd.notna(specificity)
                else np.nan
            )

            performance_rows.append({

                "Region": region,
                "Model": model_name,
                "Measurement period": period,

                "Observations": len(group),
                "Unique ponds": group["pond_id"].nunique(),

                "True positives": tp,
                "False positives": fp,
                "False negatives": fn,
                "True negatives": tn,

                "Precision (%)": precision * 100,
                "Recall (%)": recall * 100,
                "Specificity (%)": specificity * 100,
                "False positive rate (%)": false_positive_rate * 100,
                "F1-score": f1,
                "Balanced accuracy": balanced_accuracy,

                "Alert rate (%)": predicted.mean() * 100

            })

period_performance_summary = pd.DataFrame(
    performance_rows
)

numeric_columns = [
    "Precision (%)",
    "Recall (%)",
    "Specificity (%)",
    "False positive rate (%)",
    "F1-score",
    "Balanced accuracy",
    "Alert rate (%)"
]

period_performance_summary[numeric_columns] = (
    period_performance_summary[numeric_columns].round(4)
)

display(period_performance_summary)

,Region,Model,Measurement period,Observations,Unique ponds,True positives,False positives,False negatives,True negatives,Precision (%),Recall (%),Specificity (%),False positive rate (%),F1-score,Balanced accuracy,Alert rate (%)
0,Eluru,Logistic Regression,Morning,1217,118,141,694,16,366,16.8862,89.8089,34.5283,65.4717,0.2843,0.6217,68.6113
1,Eluru,Logistic Regression,Later day,1014,113,50,169,105,690,22.8311,32.2581,80.3260,19.6740,0.2674,0.5629,21.5976
2,Eluru,Logistic Regression,Other hours,1,1,0,0,0,1,NaN,NaN,100.0000,0.0000,NaN,NaN,0.0000
3,Eluru,Random Forest,Morning,1217,118,102,132,55,928,43.5897,64.9682,87.5472,12.4528,0.5217,0.7626,19.2276
4,Eluru,Random Forest,Later day,1014,113,61,218,94,641,21.8638,39.3548,74.6217,25.3783,0.2811,0.5699,27.5148
5,Eluru,Random Forest,Other hours,1,1,0,1,0,0,0.0000,NaN,0.0000,100.0000,NaN,NaN,100.0000
6,Nellore,Logistic Regression,Morning,308,49,14,9,8,277,60.8696,63.6364,96.8531,3.1469,0.6222,0.8024,7.4675
7,Nellore,Logistic Regression,Later day,265,47,3,1,19,242,75.0000,13.6364,99.5885,0.4115,0.2308,0.5661,1.5094
8,Nellore,Random Forest,Morning,308,49,21,17,1,269,55.2632,95.4545,94.0559,5.9441,0.7000,0.9476,12.3377
9,Nellore,Random Forest,Later day,265,47,0,4,22,239,0.0000,0.0000,98.3539,1.6461,NaN,0.4918,1.5094


In [28]:
# CELL 28: WATER-QUALITY PROFILE OF FALSE ALERTS AND CORRECT IN-RANGE CASES

false_alert_data = period_performance_data[
    (period_performance_data["Actual class"] == 0)
    & period_performance_data["Measurement period"].isin(
        ["Morning", "Later day"]
    )
].copy()

model_configurations = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

predictors = [
    "pH",
    "DO (mg/L)",
    "Temp (in °C)"
]

false_alert_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = false_alert_data[
        false_alert_data["Region"] == region
    ].copy()

    for model_name, prediction_column in model_configurations.items():

        region_data["Prediction outcome"] = np.where(
            region_data[prediction_column] == 1,
            "False Positive",
            "True Negative"
        )

        summary = (
            region_data
            .groupby(
                ["Measurement period", "Prediction outcome"]
            )
            .agg(
                Observations=("pond_id", "size"),
                Unique_ponds=("pond_id", "nunique"),
                Median_pH=("pH", "median"),
                Median_DO=("DO (mg/L)", "median"),
                Median_temperature=("Temp (in °C)", "median")
            )
            .reset_index()
        )

        summary["Region"] = region
        summary["Model"] = model_name

        false_alert_rows.append(summary)

false_alert_profile = pd.concat(
    false_alert_rows,
    ignore_index=True
)

false_alert_profile = false_alert_profile[
    [
        "Region",
        "Model",
        "Measurement period",
        "Prediction outcome",
        "Observations",
        "Unique_ponds",
        "Median_pH",
        "Median_DO",
        "Median_temperature"
    ]
].rename(columns={
    "Unique_ponds": "Unique ponds",
    "Median_pH": "Median pH",
    "Median_DO": "Median DO (mg/L)",
    "Median_temperature": "Median temperature (°C)"
})

numeric_columns = [
    "Median pH",
    "Median DO (mg/L)",
    "Median temperature (°C)"
]

false_alert_profile[numeric_columns] = (
    false_alert_profile[numeric_columns].round(2)
)

display(false_alert_profile)

,Region,Model,Measurement period,Prediction outcome,Observations,Unique ponds,Median pH,Median DO (mg/L),Median temperature (°C)
0,Eluru,Logistic Regression,Later day,False Positive,169,79,8.47,10.60,32.1
1,Eluru,Logistic Regression,Later day,True Negative,690,112,8.39,11.10,29.4
2,Eluru,Logistic Regression,Morning,False Positive,694,116,8.21,3.30,29.6
3,Eluru,Logistic Regression,Morning,True Negative,366,107,8.04,3.40,24.9
4,Eluru,Random Forest,Later day,False Positive,218,94,8.46,11.40,30.8
5,Eluru,Random Forest,Later day,True Negative,641,112,8.40,10.90,30.2
6,Eluru,Random Forest,Morning,False Positive,132,72,8.23,3.10,29.6
7,Eluru,Random Forest,Morning,True Negative,928,118,8.17,3.40,27.9
8,Nellore,Logistic Regression,Later day,False Positive,1,1,8.97,8.30,34.6
9,Nellore,Logistic Regression,Later day,True Negative,242,47,8.32,9.40,32.8


In [29]:
# CELL 29: MATCHED FALSE-POSITIVE VS TRUE-NEGATIVE ANALYSIS

rng = np.random.default_rng(42)
n_bootstrap = 2000

predictors = [
    "pH",
    "DO (mg/L)",
    "Temp (in °C)"
]

model_configurations = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

analysis_data = error_analysis_data.copy()

# Recover measurement period and calendar month.
analysis_data["Observation timestamp"] = pd.to_datetime(
    analysis_data["Observation timestamp"],
    errors="coerce"
)

analysis_data["Measurement hour"] = (
    analysis_data["Observation timestamp"].dt.hour
)

analysis_data["Measurement period"] = np.select(
    [
        analysis_data["Measurement hour"].between(6, 8),
        analysis_data["Measurement hour"].between(15, 17)
    ],
    ["Morning", "Later day"],
    default="Other hours"
)

analysis_data["Calendar month"] = (
    analysis_data["Observation timestamp"]
    .dt.strftime("%Y-%m")
)

matched_false_alert_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = analysis_data[
        (analysis_data["Region"] == region)
        & (analysis_data["Actual class"] == 0)
        & analysis_data["Measurement period"].isin(
            ["Morning", "Later day"]
        )
    ].copy()

    for model_name, prediction_column in model_configurations.items():

        region_data["Error outcome"] = np.where(
            region_data[prediction_column] == 1,
            "False Positive",
            "True Negative"
        )

        for period in ["Morning", "Later day"]:

            period_data = region_data[
                region_data["Measurement period"] == period
            ].copy()

            for feature in predictors:

                # Calculate median feature values by pond-month and outcome.
                pond_month_summary = (
                    period_data
                    .groupby(
                        ["pond_id", "Calendar month", "Error outcome"]
                    )[feature]
                    .median()
                    .unstack("Error outcome")
                )

                # Retain pond-months containing both outcomes.
                if not {
                    "False Positive",
                    "True Negative"
                }.issubset(pond_month_summary.columns):
                    continue

                paired = pond_month_summary.dropna(
                    subset=["False Positive", "True Negative"]
                ).copy()

                if paired.empty:
                    continue

                # Positive difference means higher values in false alerts.
                paired["Difference"] = (
                    paired["False Positive"]
                    - paired["True Negative"]
                )

                # Average within each pond to avoid giving ponds
                # with more recorded months excessive weight.
                pond_differences = (
                    paired["Difference"]
                    .groupby(level="pond_id")
                    .mean()
                )

                differences = pond_differences.to_numpy()

                # Bootstrap whole ponds.
                if len(differences) >= 2:

                    bootstrap_means = []

                    for _ in range(n_bootstrap):

                        sampled = rng.choice(
                            differences,
                            size=len(differences),
                            replace=True
                        )

                        bootstrap_means.append(sampled.mean())

                    ci_lower = np.percentile(
                        bootstrap_means, 2.5
                    )

                    ci_upper = np.percentile(
                        bootstrap_means, 97.5
                    )

                else:
                    ci_lower = np.nan
                    ci_upper = np.nan

                matched_false_alert_rows.append({

                    "Region": region,
                    "Model": model_name,
                    "Measurement period": period,
                    "Parameter": feature,

                    "Matched pond-months": len(paired),
                    "Unique ponds": len(differences),

                    "Mean pond-level difference (FP - TN)": (
                        differences.mean()
                    ),

                    "Median pond-level difference (FP - TN)": (
                        np.median(differences)
                    ),

                    "95% CI lower": ci_lower,
                    "95% CI upper": ci_upper

                })

matched_false_alert_analysis = pd.DataFrame(
    matched_false_alert_rows
)

numeric_columns = [
    "Mean pond-level difference (FP - TN)",
    "Median pond-level difference (FP - TN)",
    "95% CI lower",
    "95% CI upper"
]

matched_false_alert_analysis[numeric_columns] = (
    matched_false_alert_analysis[numeric_columns].round(3)
)

display(matched_false_alert_analysis)


,Region,Model,Measurement period,Parameter,Matched pond-months,Unique ponds,Mean pond-level difference (FP - TN),Median pond-level difference (FP - TN),95% CI lower,95% CI upper
0,Eluru,Logistic Regression,Morning,pH,42,35,0.142,0.140,0.110,0.173
1,Eluru,Logistic Regression,Morning,DO (mg/L),42,35,-0.314,-0.200,-0.558,-0.100
2,Eluru,Logistic Regression,Morning,Temp (in °C),42,35,0.587,0.450,0.141,1.024
3,Eluru,Logistic Regression,Later day,pH,59,42,0.191,0.118,0.129,0.258
4,Eluru,Logistic Regression,Later day,DO (mg/L),59,42,-0.784,-0.379,-1.417,-0.277
5,Eluru,Logistic Regression,Later day,Temp (in °C),59,42,0.612,0.562,0.236,0.992
6,Eluru,Random Forest,Morning,pH,58,40,0.009,0.012,-0.025,0.041
7,Eluru,Random Forest,Morning,DO (mg/L),58,40,-0.817,-0.625,-1.062,-0.594
8,Eluru,Random Forest,Morning,Temp (in °C),58,40,0.041,-0.125,-0.182,0.284
9,Eluru,Random Forest,Later day,pH,91,56,0.141,0.067,0.090,0.195


In [30]:
# CELL 30: PREDICTION SCORE DISTANCES FOR FALSE ALERTS AND TRUE NEGATIVES

score_data = error_analysis_data.copy()

score_data["Observation timestamp"] = pd.to_datetime(
    score_data["Observation timestamp"],
    errors="coerce"
)

score_data["Measurement hour"] = (
    score_data["Observation timestamp"].dt.hour
)

score_data["Measurement period"] = np.select(
    [
        score_data["Measurement hour"].between(6, 8),
        score_data["Measurement hour"].between(15, 17)
    ],
    ["Morning", "Later day"],
    default="Other hours"
)

# Consider observations that were actually in range.
score_data = score_data[
    (score_data["Actual class"] == 0)
    & score_data["Measurement period"].isin(
        ["Morning", "Later day"]
    )
].copy()

model_configurations = {
    "Logistic Regression": {
        "prediction": "LR predicted class",
        "probability": "LR probability",
        "threshold": "LR threshold"
    },
    "Random Forest": {
        "prediction": "RF predicted class",
        "probability": "RF probability",
        "threshold": "RF threshold"
    }
}

score_margin_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = score_data[
        score_data["Region"] == region
    ].copy()

    for model_name, config in model_configurations.items():

        prediction_column = config["prediction"]
        probability_column = config["probability"]
        threshold_column = config["threshold"]

        for period in ["Morning", "Later day"]:

            period_data = region_data[
                region_data["Measurement period"] == period
            ].copy()

            for outcome, predicted_class in [
                ("False Positive", 1),
                ("True Negative", 0)
            ]:

                group = period_data[
                    period_data[prediction_column] == predicted_class
                ].copy()

                if group.empty:
                    continue

                threshold = group[threshold_column].iloc[0]

                if predicted_class == 1:
                    distance = (
                        group[probability_column] - threshold
                    )
                else:
                    distance = (
                        threshold - group[probability_column]
                    )

                score_margin_rows.append({

                    "Region": region,
                    "Model": model_name,
                    "Measurement period": period,
                    "Prediction outcome": outcome,

                    "Observations": len(group),
                    "Unique ponds": group["pond_id"].nunique(),

                    "Threshold": threshold,

                    "Mean distance": distance.mean(),
                    "Median distance": distance.median(),

                    "25th percentile": distance.quantile(0.25),
                    "75th percentile": distance.quantile(0.75),

                    "Within 0.05 of threshold (%)": (
                        distance.le(0.05).mean() * 100
                    ),

                    "Within 0.10 of threshold (%)": (
                        distance.le(0.10).mean() * 100
                    )

                })

false_alert_margin_analysis = pd.DataFrame(
    score_margin_rows
)

numeric_columns = [
    "Mean distance",
    "Median distance",
    "25th percentile",
    "75th percentile",
    "Within 0.05 of threshold (%)",
    "Within 0.10 of threshold (%)"
]

false_alert_margin_analysis[numeric_columns] = (
    false_alert_margin_analysis[numeric_columns].round(4)
)

display(false_alert_margin_analysis)

,Region,Model,Measurement period,Prediction outcome,Observations,Unique ponds,Threshold,Mean distance,Median distance,25th percentile,75th percentile,Within 0.05 of threshold (%),Within 0.10 of threshold (%)
0,Eluru,Logistic Regression,Morning,False Positive,694,116,0.2,0.0612,0.0547,0.0322,0.0819,43.3718,85.5908
1,Eluru,Logistic Regression,Morning,True Negative,366,107,0.2,0.0418,0.0359,0.0180,0.0619,66.6667,94.2623
2,Eluru,Logistic Regression,Later day,False Positive,169,79,0.2,0.0496,0.0289,0.0121,0.0617,66.2722,84.0237
3,Eluru,Logistic Regression,Later day,True Negative,690,112,0.2,0.0504,0.0454,0.0255,0.0718,54.7826,90.5797
4,Eluru,Random Forest,Morning,False Positive,132,72,0.4,0.1944,0.1342,0.0506,0.3536,25.0000,43.1818
5,Eluru,Random Forest,Morning,True Negative,928,118,0.4,0.2360,0.2480,0.1675,0.3181,4.2026,12.2845
6,Eluru,Random Forest,Later day,False Positive,218,94,0.4,0.1015,0.0833,0.0337,0.1445,38.0734,60.5505
7,Eluru,Random Forest,Later day,True Negative,641,112,0.4,0.1368,0.1245,0.0623,0.2045,19.5008,39.4696
8,Nellore,Logistic Regression,Morning,False Positive,9,8,0.4,0.0163,0.0096,0.0076,0.0299,100.0000,100.0000
9,Nellore,Logistic Regression,Morning,True Negative,277,49,0.4,0.1135,0.1052,0.0639,0.1596,17.3285,48.3755


In [31]:
# CELL 31: MODEL PERFORMANCE BY EXACT MEASUREMENT HOUR

from sklearn.metrics import confusion_matrix

hourly_data = error_analysis_data.copy()

hourly_data["Observation timestamp"] = pd.to_datetime(
    hourly_data["Observation timestamp"],
    errors="coerce"
)

hourly_data["Measurement hour"] = (
    hourly_data["Observation timestamp"].dt.hour
)

# Focus on the principal recorded measurement hours.
selected_hours = [6, 7, 8, 15, 16, 17]

hourly_data = hourly_data[
    hourly_data["Measurement hour"].isin(selected_hours)
].copy()

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

hourly_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = hourly_data[
        hourly_data["Region"] == region
    ]

    for model_name, prediction_column in model_columns.items():

        for hour in selected_hours:

            group = region_data[
                region_data["Measurement hour"] == hour
            ]

            if group.empty:
                continue

            actual = group["Actual class"].astype(int)
            predicted = group[prediction_column].astype(int)

            tn, fp, fn, tp = confusion_matrix(
                actual,
                predicted,
                labels=[0, 1]
            ).ravel()

            precision = (
                tp / (tp + fp)
                if tp + fp else np.nan
            )

            recall = (
                tp / (tp + fn)
                if tp + fn else np.nan
            )

            specificity = (
                tn / (tn + fp)
                if tn + fp else np.nan
            )

            f1 = (
                2 * precision * recall / (precision + recall)
                if (
                    pd.notna(precision)
                    and pd.notna(recall)
                    and precision + recall > 0
                )
                else np.nan
            )

            balanced_accuracy = (
                (recall + specificity) / 2
                if pd.notna(recall) and pd.notna(specificity)
                else np.nan
            )

            hourly_rows.append({

                "Region": region,
                "Model": model_name,
                "Measurement hour": f"{hour:02d}:00",

                "Observations": len(group),
                "Unique ponds": group["pond_id"].nunique(),

                "Out-of-range cases": int(actual.sum()),

                "True positives": tp,
                "False positives": fp,
                "False negatives": fn,
                "True negatives": tn,

                "Precision (%)": precision * 100,
                "Recall (%)": recall * 100,
                "Specificity (%)": specificity * 100,

                "F1-score": f1,
                "Balanced accuracy": balanced_accuracy,

                "Alert rate (%)": predicted.mean() * 100

            })

hourly_performance_summary = pd.DataFrame(hourly_rows)

numeric_columns = [
    "Precision (%)",
    "Recall (%)",
    "Specificity (%)",
    "F1-score",
    "Balanced accuracy",
    "Alert rate (%)"
]

hourly_performance_summary[numeric_columns] = (
    hourly_performance_summary[numeric_columns].round(4)
)

display(hourly_performance_summary)

,Region,Model,Measurement hour,Observations,Unique ponds,Out-of-range cases,True positives,False positives,False negatives,True negatives,Precision (%),Recall (%),Specificity (%),F1-score,Balanced accuracy,Alert rate (%)
0,Eluru,Logistic Regression,06:00,81,53,17,15,55,2,9,21.4286,88.2353,14.0625,0.3448,0.5115,86.4198
1,Eluru,Logistic Regression,07:00,960,116,126,114,557,12,277,16.9896,90.4762,33.2134,0.2861,0.6184,69.8958
2,Eluru,Logistic Regression,08:00,176,85,14,12,82,2,80,12.7660,85.7143,49.3827,0.2222,0.6755,53.4091
3,Eluru,Logistic Regression,15:00,287,103,47,8,16,39,224,33.3333,17.0213,93.3333,0.2254,0.5518,8.3624
4,Eluru,Logistic Regression,16:00,707,112,108,42,143,66,456,22.7027,38.8889,76.1269,0.2867,0.5751,26.1669
5,Eluru,Logistic Regression,17:00,20,16,0,0,10,0,10,0.0000,NaN,50.0000,NaN,NaN,50.0000
6,Eluru,Random Forest,06:00,81,53,17,9,22,8,42,29.0323,52.9412,65.6250,0.3750,0.5928,38.2716
7,Eluru,Random Forest,07:00,960,116,126,86,96,40,738,47.2527,68.2540,88.4892,0.5584,0.7837,18.9583
8,Eluru,Random Forest,08:00,176,85,14,7,14,7,148,33.3333,50.0000,91.3580,0.4000,0.7068,11.9318
9,Eluru,Random Forest,15:00,287,103,47,18,57,29,183,24.0000,38.2979,76.2500,0.2951,0.5727,26.1324


In [32]:
# CELL 32: POND-LEVEL BOOTSTRAP — EXACT-HOUR RECALL

rng = np.random.default_rng(42)
n_bootstrap = 2000

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

hourly_bootstrap_rows = []

hours_to_analyse = [6, 7, 8, 15, 16, 17]

for region in ["Eluru", "Nellore"]:

    region_data = error_analysis_data[
        error_analysis_data["Region"] == region
    ].copy()

    region_data["Observation timestamp"] = pd.to_datetime(
        region_data["Observation timestamp"],
        errors="coerce"
    )

    region_data["Measurement hour"] = (
        region_data["Observation timestamp"].dt.hour
    )

    for model_name, prediction_column in model_columns.items():

        for hour in hours_to_analyse:

            group = region_data[
                region_data["Measurement hour"] == hour
            ].copy()

            if group.empty:
                continue

            group["True positive"] = (
                (group["Actual class"] == 1)
                & (group[prediction_column] == 1)
            ).astype(int)

            # Aggregate positive cases and detections within each pond.
            pond_counts = (
                group.groupby("pond_id")
                .agg(
                    actual_positive=("Actual class", "sum"),
                    true_positive=("True positive", "sum")
                )
            )

            actual_positives = int(
                pond_counts["actual_positive"].sum()
            )

            detected_positives = int(
                pond_counts["true_positive"].sum()
            )

            unique_ponds_with_positives = int(
                (pond_counts["actual_positive"] > 0).sum()
            )

            if actual_positives == 0:

                observed_recall = np.nan
                ci_lower = np.nan
                ci_upper = np.nan
                valid_bootstrap = 0

            else:

                observed_recall = (
                    detected_positives / actual_positives
                )

                positive_counts = (
                    pond_counts["actual_positive"].to_numpy()
                )

                detected_counts = (
                    pond_counts["true_positive"].to_numpy()
                )

                bootstrap_recall = []

                for _ in range(n_bootstrap):

                    sampled_indices = rng.integers(
                        0,
                        len(pond_counts),
                        size=len(pond_counts)
                    )

                    sampled_positives = (
                        positive_counts[sampled_indices].sum()
                    )

                    if sampled_positives == 0:
                        continue

                    sampled_detected = (
                        detected_counts[sampled_indices].sum()
                    )

                    bootstrap_recall.append(
                        sampled_detected / sampled_positives
                    )

                valid_bootstrap = len(bootstrap_recall)

                ci_lower = np.percentile(
                    bootstrap_recall, 2.5
                )

                ci_upper = np.percentile(
                    bootstrap_recall, 97.5
                )

            hourly_bootstrap_rows.append({

                "Region": region,
                "Model": model_name,
                "Measurement hour": f"{hour:02d}:00",

                "Observations": len(group),
                "Unique ponds": group["pond_id"].nunique(),

                "Actual out-of-range cases": actual_positives,

                "Ponds with out-of-range cases": (
                    unique_ponds_with_positives
                ),

                "Detected cases": detected_positives,

                "Recall (%)": (
                    observed_recall * 100
                    if pd.notna(observed_recall)
                    else np.nan
                ),

                "95% CI lower (%)": (
                    ci_lower * 100
                    if pd.notna(ci_lower)
                    else np.nan
                ),

                "95% CI upper (%)": (
                    ci_upper * 100
                    if pd.notna(ci_upper)
                    else np.nan
                ),

                "Valid bootstrap samples": valid_bootstrap

            })

hourly_bootstrap_recall = pd.DataFrame(
    hourly_bootstrap_rows
)

numeric_columns = [
    "Recall (%)",
    "95% CI lower (%)",
    "95% CI upper (%)"
]

hourly_bootstrap_recall[numeric_columns] = (
    hourly_bootstrap_recall[numeric_columns].round(2)
)

display(hourly_bootstrap_recall)

,Region,Model,Measurement hour,Observations,Unique ponds,Actual out-of-range cases,Ponds with out-of-range cases,Detected cases,Recall (%),95% CI lower (%),95% CI upper (%),Valid bootstrap samples
0,Eluru,Logistic Regression,06:00,81,53,17,13,15,88.24,70.59,100.00,2000
1,Eluru,Logistic Regression,07:00,960,116,126,56,114,90.48,82.44,97.35,2000
2,Eluru,Logistic Regression,08:00,176,85,14,10,12,85.71,66.67,100.00,2000
3,Eluru,Logistic Regression,15:00,287,103,47,30,8,17.02,2.70,33.33,2000
4,Eluru,Logistic Regression,16:00,707,112,108,49,42,38.89,25.92,51.18,2000
5,Eluru,Logistic Regression,17:00,20,16,0,0,0,NaN,NaN,NaN,0
6,Eluru,Random Forest,06:00,81,53,17,13,9,52.94,25.00,80.00,2000
7,Eluru,Random Forest,07:00,960,116,126,56,86,68.25,56.92,78.69,2000
8,Eluru,Random Forest,08:00,176,85,14,10,7,50.00,16.67,70.59,2000
9,Eluru,Random Forest,15:00,287,103,47,30,18,38.30,26.00,50.00,2000


In [33]:
# CELL 33: CONSOLIDATED MORNING VS LATER-DAY ERROR ANALYSIS

hourly_error_data = hourly_data.copy()

hourly_error_data["Measurement period"] = np.select(
    [
        hourly_error_data["Measurement hour"].between(6, 8),
        hourly_error_data["Measurement hour"].between(15, 17)
    ],
    ["Morning (06:00–08:59)", "Later day (15:00–17:59)"],
    default="Other hours"
)

hourly_error_data = hourly_error_data[
    hourly_error_data["Measurement period"].isin(
        ["Morning (06:00–08:59)", "Later day (15:00–17:59)"]
    )
].copy()

model_columns = {
    "Logistic Regression": "LR predicted class",
    "Random Forest": "RF predicted class"
}

consolidated_error_rows = []

for region in ["Eluru", "Nellore"]:

    region_data = hourly_error_data[
        hourly_error_data["Region"] == region
    ]

    for model_name, prediction_column in model_columns.items():

        for period in [
            "Morning (06:00–08:59)",
            "Later day (15:00–17:59)"
        ]:

            group = region_data[
                region_data["Measurement period"] == period
            ]

            actual = group["Actual class"].astype(int)
            predicted = group[prediction_column].astype(int)

            tn, fp, fn, tp = confusion_matrix(
                actual,
                predicted,
                labels=[0, 1]
            ).ravel()

            consolidated_error_rows.append({

                "Region": region,
                "Model": model_name,
                "Measurement period": period,

                "Observations": len(group),
                "Unique ponds": group["pond_id"].nunique(),

                "Actual out-of-range": int(actual.sum()),

                "Detected problems": tp,
                "Missed problems": fn,

                "False alerts": fp,
                "Correct in-range predictions": tn,

                "Recall (%)": (
                    tp / (tp + fn) * 100
                    if tp + fn else np.nan
                ),

                "False positive rate (%)": (
                    fp / (fp + tn) * 100
                    if fp + tn else np.nan
                ),

                "Alert rate (%)": predicted.mean() * 100

            })

consolidated_hourly_errors = pd.DataFrame(
    consolidated_error_rows
)

numeric_columns = [
    "Recall (%)",
    "False positive rate (%)",
    "Alert rate (%)"
]

consolidated_hourly_errors[numeric_columns] = (
    consolidated_hourly_errors[numeric_columns].round(2)
)

display(consolidated_hourly_errors)

,Region,Model,Measurement period,Observations,Unique ponds,Actual out-of-range,Detected problems,Missed problems,False alerts,Correct in-range predictions,Recall (%),False positive rate (%),Alert rate (%)
0,Eluru,Logistic Regression,Morning (06:00–08:59),1217,118,157,141,16,694,366,89.81,65.47,68.61
1,Eluru,Logistic Regression,Later day (15:00–17:59),1014,113,155,50,105,169,690,32.26,19.67,21.60
2,Eluru,Random Forest,Morning (06:00–08:59),1217,118,157,102,55,132,928,64.97,12.45,19.23
3,Eluru,Random Forest,Later day (15:00–17:59),1014,113,155,61,94,218,641,39.35,25.38,27.51
4,Nellore,Logistic Regression,Morning (06:00–08:59),308,49,22,14,8,9,277,63.64,3.15,7.47
5,Nellore,Logistic Regression,Later day (15:00–17:59),265,47,22,3,19,1,242,13.64,0.41,1.51
6,Nellore,Random Forest,Morning (06:00–08:59),308,49,22,21,1,17,269,95.45,5.94,12.34
7,Nellore,Random Forest,Later day (15:00–17:59),265,47,22,0,22,4,239,0.00,1.65,1.51


# Final Interpretation and Conclusions

## 1. Purpose

This notebook examined the behaviour of the Logistic Regression and Random Forest models developed in the predictive modelling stage.

The analysis focused on prediction errors, pond-level error patterns, measurement timing, water-quality characteristics, prediction probabilities and classification performance.

The objective was to understand not only how frequently the models made incorrect predictions, but also where and under what observed conditions those errors occurred.

## 2. Prediction error patterns

Both models produced false positives and false negatives across Eluru and Nellore.

The error analysis showed that:

- Prediction errors were distributed across multiple ponds rather than being restricted to a single pond.
- Some ponds repeatedly experienced missed out-of-range observations.
- Logistic Regression generally produced more false alerts in Eluru.
- Random Forest reduced false alerts in several comparisons, although it also missed some actual out-of-range observations.

These results demonstrate that model performance depends on the error type being considered.

## 3. Measurement timing and model performance

Classification performance differed between observations recorded in the morning and later in the day.

When grouped by the earlier measurement timestamp, both models showed substantially different recall across these periods.

However, a subsequent investigation using the next observation timestamp revealed that future measurement time and elapsed observation interval were strongly associated.

The available observations did not provide sufficient overlap to isolate the independent effect of measurement time.

Therefore, the observed differences should be interpreted as differences in predictive performance across recorded monitoring conditions, not as evidence that one measurement period inherently makes water-quality problems easier or harder to predict.

## 4. Pond-level analysis

Repeated observations from the same pond were examined to understand whether prediction errors persisted within individual ponds.

Paired pond-level comparisons showed that detection performance varied between morning and later-day observations.

Pond-level bootstrap confidence intervals supported the observed differences in several comparisons.

However, the number of ponds and actual out-of-range observations varied across regions and subgroups. Results from smaller subgroups require particular caution.

## 5. Observation interval and predictive performance

The enriched prediction dataset allowed the earlier measurement timestamp to be distinguished from the next observation timestamp.

The elapsed intervals were highly irregular.

The interval-based analysis showed:

- Short-interval predictions generally provided stronger detection performance.
- Random Forest achieved better balanced accuracy for short intervals in both regions.
- Logistic Regression achieved higher recall than Random Forest in Eluru's short-interval subgroup.
- Both models demonstrated weak detection performance for long intervals, particularly in Nellore.

Pond-clustered bootstrap comparisons provided additional evidence about model differences.

These findings are observational. They do not establish that increasing the observation interval causes predictive performance to deteriorate.

## 6. Probability and threshold behaviour

The analysis of predicted probabilities and distances from classification thresholds helped distinguish confidently missed cases from observations close to the decision boundary.

This provided additional insight into model errors beyond the final predicted class.

Probability scores were interpreted within their respective models and were not assumed to be directly comparable across Logistic Regression and Random Forest.

## 7. Limitations

The interpretation of these results is subject to the following limitations:

1. The dataset contains repeated observations from the same ponds.
2. Observation intervals are irregular and can extend over several months.
3. Future measurement time and elapsed observation interval have limited overlap.
4. Some subgroup analyses contain relatively few actual out-of-range observations.
5. Recorded water-quality classifications may reflect changes in monitoring and classification practices.
6. The recorded measurements do not establish causal relationships.
7. Performance on the available test partitions does not guarantee equivalent performance on new farms or future monitoring periods.

## 8. Overall conclusion

The error analysis demonstrates that AQUA's predictive models have different strengths and weaknesses depending on the region, observation conditions and error metric.

Random Forest offers stronger balanced performance in several short-interval settings, while Logistic Regression retains advantages in recall under some conditions.

Both models have limitations, particularly when predicting the status of observations separated by long intervals.

The findings reinforce the importance of considering pond-level dependence, monitoring frequency and observation timing when interpreting predictive performance.

This notebook provides a retrospective investigation of model behaviour and error patterns. Its findings support further validation and refinement before the models are considered suitable for operational aquaculture decision support.